# ML3. Validation – Оценка качества модели

## **Задание 1. Ответьте на вопросы из введения**

### ⬇️ Задание 1.1. *Что такое leave-one-out? Укажите ограничения и сильные стороны.*

**Leave-one-out** наверное, самый объективный подход к оценке переобученности модели.  
Из исходных выборок, которые у нас есть, мы убираем поочерёдно по одному вектору наблюдения и 
обучаем каждый раз на оставшейся выборке, на оставшейся части. В итоге у нас получится некая 
первая модель. Далее "возвращаем первую выборку назад", вытаскиваем другой вектор и получается вторая модель. 
И так поочерёдно по одному объекту мы убираем из обучающей выборки и настраиваем модель по оставшимся наблюдениям. 
По итогу у нас получается `N` различных моделей, где `N` - это количество объектов в обучающей выборке.  
Затем, на основе всех полученных `N` моделей формирутся общая модель, которая бы обладала лучшими обобщающими способностями, т.е. получаем `N` значений
ошибки и берём их среднее – это и есть итоговая оценка качества. 

- Ограничения:  
    - супер трудоёмкий процесс для большого количества значений N. Каждая моделька N считается отдельно. 
    В текущем датасете ~50к обучений, а не 5. Здесь это неприменимо.
    - невозможна стратификация т.к. в фолде из одного объекта нельзя сохранить доли классов.
    - обучающие выборки соседних шагов отличаются всего одним объектом,
    поэтому модели почти одинаковые, их ошибки сильно связаны между собой, и усреднение плохо
    гасит шум. Из-за этого разброс итоговой оценки оказывается больше, чем у 5- или 10-фолда.

- Сильные стороны: 
    - т.к. модель обучается почти на всех данных, то оценка качества улучшается и снижается переобучение
    - каждый объект ровно один раз побывал в проверочной части


**Вывод:** На маленьких выборках (десятки наблюдений – медицина, лабораторные опыты), где каждое наблюдение важно, а обучение модели стоит дёшево. 
В остальных случаях разумный компромисс – `KFold` с `k = 5` или `k = 10`: качество оценки почти такое же, а времени тратится в тысячи раз меньше.

### ⬇️ Задание 1.2. *Как работают Grid Search, Randomized Grid Search и байесовская оптимизация?*

Все три решают задачу подбора гиперпараметров по типу глубины дерева или силе регуляризации.

- **Grid Search (поиск по сетке)**  
Мы задаём список значений для каждого гиперпараметра где перебираются все комбинации – например, пять значений `alpha` и три значения `l1_ratio` – и метод перебирает все их сочетания, то есть 15 комбинаций.


- **Randomized Search (случайный поиск)**  
Вместо списком мы задаём диапазоны, и говорим «сделай N попыток». И метод каждый раз берёт случайную точку из этих диапазонов и находит лучшее решение.


- **Байесовская оптимизация**  
"Умный метод" поиска лучших параметров который экономит время используя прошлые результаты для каждого нового шага: 
    - Суррогатная модель - создаёт приблизительную карту вероятности на основе прошлых тестов
    - Чёрный ящик - целевая задача, где нам неизвестно математическое уравнение, а расчёты занимают много времени.

Сначала пробует несколько точек чтобы было от чего оттолкнуться и по этим результатам строит вспомогательную модель качества (приблизительную карту), затем выбирает следующую точку так, чтобы сбалансировать: пойти в место где хороший результат или где мы ничего не знаем и найти сюрприз.


![картинка](../materials/grid_vs_random_vs_bayesian_search.png)

### ⬇️ Задание 1.3. *Объясните классификацию методов отбора признаков. Объясните, как работают Пирсон и Chi2. Объясните, как работает Lasso. Объясните, что такое permutation significance. Познакомьтесь с SHAP.*

Есть три основные группы: *Filter, Wrapper и Embedded.*

- **Filter** – оценивают сами признаки и их связь с целевой переменной, не обучая конкретную ML-модель. 
    - Корреляция Пирсона – смотрим, насколько сильно признаки коррелируют друг с другом. Если два признака почти одинаковые, один можно удалить. Используется для вычисления линейной зависимости между признаками.
    - Хи-квадрат – оцениваем зависимость категориального признака от целевой переменной. Чем меньше значение хи-квадрат, тем меньше зависимость. 

- **Wrapper** – обучают модель на разных наборах признаков и мы смотрим, какой набор даёт лучший результат.
    - RFE (Recursive Feature Elimination) — рекурсивное исключение признаков.
    - Forward Selection — начинаем с малого количества признаков и постепенно добавляем.
    - Backward Elimination — начинаем со всех признаков и постепенно удаляем.
    - Permutation Importance часто рассматривают как модельно-ориентированный метод отбора: после обучения перемешиваем значения признака и смотрим, насколько ухудшилась метрика.

- **Embedded** – отбор происходит сам собой внутри обучения: `Lasso` зануляет веса
  бесполезных признаков, дерево показывает `feature_importances_`. Результат привязан к конкретной модели.

![картинка2](../materials/feature_selection_taxonomy.png)

**Коэффициент корреляции Пирсона** используется для вычисления линейной взаимосвязи между признаками. 
Это число от −1 до 1, показывающее, насколько хорошо связь между двумя
величинами описывается прямой линией. 0 означает отсутствие линейной связи, а знак говорит о направлении. Для отбора признаков берут модуль корреляции каждого признака с целевой переменной и оставляют самые высокие.

Пирсон видит *только линейную связь*: если зависимость параболическая, корреляция может быть ровно ноль при идеальной предсказуемости. Он чувствителен к выбросам — одна аномальная точка способна поднять или убить корреляцию.

**Критерий хи-квадрат** позволяет оценить, связаны ли между собой два категориальных признака.  
Какая логика: строим таблицу, сколько объектов попало в каждое сочетание *«значение признака × класс»*. Затем считаем, сколько объектов оказалось бы в каждой ячейке, если бы признак и класс были совершенно независимы (то есть просто по общим долям). Чем сильнее реальные числа расходятся с этими ожидаемыми, тем сильнее связь. Признаки сортируют по величине расхождения.

**Lasso.** Это линейная регрессия, к ошибке которой добавлен штраф за сумму модулей весов.
Модель одновременно старается и предсказывать точнее, и держать веса маленькими. Ключевая
особенность именно такого штрафа: он «поджимает» веса к нулю и для слабых признаков делает вес
**ровно нулём**, а не просто маленьким. Признак с нулевым весом на предсказание никак не влияет —
значит, модель отобрала признаки сама. Чем больше `alpha` (сила штрафа), тем больше нулей и тем
короче итоговый набор.  
Родственный `Ridge` штрафует сумму квадратов весов — он тоже уменьшает
веса, но в ноль их не обращает, поэтому сам по себе признаки не отбирает; зато он лучше ведёт
себя, когда признаки сильно скоррелированы: `Ridge` распределит вес между ними, а `Lasso`
оставит одного случайного представителя группы. Компромисс между ними — `ElasticNet`
с параметром `l1_ratio`. Важнейшее практическое условие: перед `Lasso` признаки обязательно
**нормализуют**. Штраф общий для всех весов, а вес зависит от масштаба признака, поэтому без
нормализации признак, измеренный в крупных единицах, получит маленький вес и будет занулён
незаслуженно.

**Permutation importance (значимость через перемешивание):**   
Считается **после** того, как модель
обучена. Берём один признак и случайно перемешиваем его значения между объектами: сам столбец
остаётся тем же по составу, но его связь с целевой переменной разрушается. Заново считаем
метрику. Насколько качество просело — настолько признак и был важен. Если модель не заметила
подмены, признак ей не нужен. Процедуру повторяют несколько раз и результаты усредняют, а разброс
между повторами показывает, насколько оценке можно доверять. Метод подкупает тем, что не зависит
от устройства модели (работает с чем угодно) и позволяет мерить важность в терминах той метрики,
которая нам действительно нужна. Две оговорки: считать надо на **отложенной выборке**, а не на
обучающей — иначе мы измерим, насколько модель запомнила обучающие данные, а не насколько признак
полезен; и при сильно скоррелированных признаках важность занижается у обоих, потому что модель
компенсирует испорченный признак его двойником.

**SHAP:**   
Метод отвечает на вопрос «почему модель выдала именно такое предсказание для этого
конкретного объекта». Идея заимствована из теории игр: предсказание — это общий выигрыш команды,
а признаки — игроки, и надо честно разделить выигрыш между ними. Честно означает: вклад игрока
считается как то, насколько он улучшает результат команды, усреднённое по всем возможным порядкам
вхождения игроков в команду. Результат — набор чисел, по одному на признак, которые в сумме
с базовым (средним) предсказанием дают ровно предсказание модели для этого объекта. Отсюда сразу
две вещи: **локальное** объяснение (разбор одного объекта: этот признак поднял цену на столько-то,
тот опустил) и **глобальная** важность признака — среднее значение модуля вклада по всем объектам.
Считать вклады точно в общем случае очень дорого, поэтому в библиотеке есть быстрые версии для
частных случаев: `LinearExplainer` для линейных моделей (там всё считается сразу и точно) и
`TreeExplainer` для деревьев и их ансамблей; универсальный `KernelExplainer` работает с любой
моделью, но приближённо и заметно медленнее.

---
## **Задание 2. Введение – выполните всю предобработку из предыдущего занятия**

### ⬇️ Задание 2.0. Импорт библиотек

In [362]:
import re
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, GroupKFold, StratifiedKFold, TimeSeriesSplit
from sklearn.inspection import permutation_importance

import shap
import optuna

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
# pd.set_option('display.width', 160)

RANDOM_STATE = 21
np.random.seed(RANDOM_STATE)

### ⬇️ Задание 2.1. *Прочитайте все данные.*

In [363]:
train_df = pd.read_json('../datasets/train.json')

print('Размер исходного датасета:', train_df.shape)
train_df.head(3)

Размер исходного датасета: (49352, 15)


,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,medium
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,low
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, Laundry in Building, Laund...",40.7575,7158677,-73.9625,c8b10a317b766204f08e613cef4ce7a0,[https://photos.renthop.com/2/7158677_c897a134...,3495,405 East 56th Street,medium


In [364]:
train_df.info()

<class 'pandas.DataFrame'>
Index: 49352 entries, 4 to 124009
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   bathrooms        49352 non-null  float64
 1   bedrooms         49352 non-null  int64  
 2   building_id      49352 non-null  str    
 3   created          49352 non-null  str    
 4   description      49352 non-null  str    
 5   display_address  49352 non-null  str    
 6   features         49352 non-null  object 
 7   latitude         49352 non-null  float64
 8   listing_id       49352 non-null  int64  
 9   longitude        49352 non-null  float64
 10  manager_id       49352 non-null  str    
 11  photos           49352 non-null  object 
 12  price            49352 non-null  int64  
 13  street_address   49352 non-null  str    
 14  interest_level   49352 non-null  str    
dtypes: float64(3), int64(3), object(2), str(7)
memory usage: 6.0+ MB


Дальше повторяем предобработку из проекта ML2 – по одному шагу на ячейку:

1. отрезаем выбросы цены,
2. `interest_level` кодируем,
3. создаём бинарные признаки из столбца `features`.

In [365]:
TARGET = 'price'            # наш таргет
DATE_FIELD = 'created'      # по этому полю делаем разбиения по времени
GROUP_FIELD = 'manager_id'  # по этому полю группируем в Group K-Fold
N_AMENITIES = 19            # сколько удобств берём из столбца features

data = train_df.copy()
data[DATE_FIELD] = pd.to_datetime(data[DATE_FIELD])
print('Период наблюдений:', data[DATE_FIELD].min(), '|', data[DATE_FIELD].max())

Период наблюдений: 2016-04-01 22:12:41 | 2016-06-29 21:41:47


Как в ML2 отрезаем хвосты по 1-му и 99-му перцентилю.

In [366]:
q_low, q_high = data[TARGET].quantile([0.01, 0.99])
print(f'1%-перцентиль = {q_low:.0f}, 99%-перцентиль = {q_high:.0f}')
print('Цена до отсечения: от', data[TARGET].min(), 'до', data[TARGET].max())
print('- - - - - - - - - - - - - - ')

data = data[(data[TARGET] >= q_low) & (data[TARGET] <= q_high)].reset_index(drop=True)
print('Цена после отсечения: от', data[TARGET].min(), 'до', data[TARGET].max())
print('Размер датасета после отсечения выбросов:', data.shape)

1%-перцентиль = 1475, 99%-перцентиль = 13000
Цена до отсечения: от 43 до 4490000
- - - - - - - - - - - - - - 
Цена после отсечения: от 1475 до 13000
Размер датасета после отсечения выбросов: (48379, 15)


**Шаг 2. Кодируем `interest_level`.** Уровень интереса к объявлению — величина порядковая
(`low` < `medium` < `high`), поэтому заменяем её числами 0/1/2. Модель работает только с числами,
строковый столбец в неё не подать.

In [367]:
data['interest_level'] = data['interest_level'].map({'low': 0, 'medium': 1, 'high': 2})

print('Значения interest_level после кодирования:', sorted(data['interest_level'].unique()))
data['interest_level'].value_counts().sort_index()

Значения interest_level после кодирования: [np.int64(0), np.int64(1), np.int64(2)]


interest_level
0    33697
1    11116
2     3566
Name: count, dtype: int64

### ⬇️ Задание 2.2. *Создайте признаки: 'Elevator', 'HardwoodFloors', 'CatsAllowed', 'DogsAllowed', 'Doorman', 'Dishwasher', 'NoFee', 'LaundryinBuilding', 'FitnessCenter', 'Pre-War', 'LaundryinUnit', 'RoofDeck', 'OutdoorSpace', 'DiningRoom', 'HighSpeedInternet', 'Balcony', 'SwimmingPool', 'NewConstruction', 'Terrace'.*

In [368]:
print('Тип значения в одной ячейке столбца:', type(data['features'].iloc[0]).__name__)
display(data['features'].head(3))

Тип значения в одной ячейке столбца: list


0    [Dining Room, Pre-War, Laundry in Building, Di...
1    [Doorman, Elevator, Laundry in Building, Dishw...
2    [Doorman, Elevator, Laundry in Building, Laund...
Name: features, dtype: object

In [369]:
# explode() разворачивает списки: одна строка = одно название
# Так все названия из всех объявлений оказываются в одной Series и их можно посчитать
raw_names = data['features'].explode().dropna()

print('Всего названий во всех объявлениях:', len(raw_names))
print('Уникальных названий (как в данных):', raw_names.nunique())
display(raw_names.value_counts().head(8))

Всего названий во всех объявлениях: 262573
Уникальных названий (как в данных): 1539


features
Elevator               25398
Hardwood Floors        23159
Cats Allowed           23148
Dogs Allowed           21662
Doorman                20497
Dishwasher             20095
No Fee                 17806
Laundry in Building    16093
Name: count, dtype: int64

Названия записаны с пробелами - Hardwood Floors, Laundry in Building. В таком виде их нельзя использовать как имена столбцов, поэтому дальше чистим

In [370]:
def clean_features(feature_list):
    # Чистка как в ML2 - 'Hardwood Floors' -> 'HardwoodFloors'
    text = str(feature_list)
    for char in ['[', ']', "'", '"', ' ']:
        text = text.replace(char, '')
    return text.split(',')

print('Было: ', data['features'].iloc[0])
print('Стало:', clean_features(data['features'].iloc[0]))

Было:  ['Dining Room', 'Pre-War', 'Laundry in Building', 'Dishwasher', 'Hardwood Floors', 'Dogs Allowed', 'Cats Allowed']
Стало: ['DiningRoom', 'Pre-War', 'LaundryinBuilding', 'Dishwasher', 'HardwoodFloors', 'DogsAllowed', 'CatsAllowed']


После чистки часть названий склеилась сама собой - например, Hardwood Floors и HardwoodFloors стали одной строкой. Сравним количество уникальных названий до и после

In [371]:
# Собираем все названия удобств из столбца features в одну Series
all_names = pd.Series([name for feature_list in data['features']
                       for name in clean_features(feature_list)])

print('Уникальных названий до чистки:  ', raw_names.nunique())
print('Уникальных названий после чистки:', all_names.nunique())
print('Склеилось за счёт удаления пробелов:', raw_names.nunique() - all_names.nunique())

Уникальных названий до чистки:   1539
Уникальных названий после чистки: 1530
Склеилось за счёт удаления пробелов: 9


Посмотрим есть ли дубли, признаки могут быть написаны по-разному, с разным регистром. Если числа разошлись - какие-то названия отличаются только регистром букв, поэтому в отдельной колонке будет записаны удобства без регистра, чтобы они были одинаковыми (.lower() даёт одно и то же)

In [372]:
print('Уникальных написаний:          ', all_names.nunique())
print('Уникальных без учёта регистра: ', all_names.str.lower().nunique())
print('Значит, написаний-дублей:      ', all_names.nunique() - all_names.str.lower().nunique())

# Смотрим, какие именно названия записаны больше чем одним способом.
variants = all_names.value_counts().rename_axis('написание').reset_index(name='встречается')
variants['без регистра'] = variants['написание'].str.lower()
# transform('size') — сколько написаний в группе, transform('sum') — суммарная частота группы
variants['написаний у названия'] = variants.groupby('без регистра')['написание'].transform('size')
variants['всего с учётом дублей'] = variants.groupby('без регистра')['встречается'].transform('sum')

duplicates = variants[variants['написаний у названия'] > 1].sort_values(
    ['всего с учётом дублей', 'встречается'], ascending=False)

print('Названий, записанных больше чем одним способом:', duplicates['без регистра'].nunique())
display(duplicates.head(30))

Уникальных написаний:           1530
Уникальных без учёта регистра:  1258
Значит, написаний-дублей:       272
Названий, записанных больше чем одним способом: 188


,написание,встречается,без регистра,написаний у названия,всего с учётом дублей
0,Elevator,25398,elevator,3,25742
44,elevator,343,elevator,3,25742
1267,ELEVATOR,1,elevator,3,25742
1,HardwoodFloors,23159,hardwoodfloors,4,23190
145,Hardwoodfloors,19,hardwoodfloors,4,23190
199,hardwoodfloors,8,hardwoodfloors,4,23190
282,HARDWOODFLOORS,4,hardwoodfloors,4,23190
4,Doorman,20497,doorman,2,20566
86,doorman,69,doorman,2,20566
5,Dishwasher,20095,dishwasher,4,20476


In [373]:
# Что наше слияние по регистру НЕ поймало: расхождения в дефисах которые нам нужны в список удобств
def show_group(substr):
    rows = [(n, c) for n, c in all_names.value_counts().items()
            if substr in n.lower().replace('-', '') and len(n) < 30]
    return pd.DataFrame(rows, columns=['написание', 'встречается'])

print('=== Pre-War: дефис мешает слиянию ===')
display(show_group('prewar'))

print('=== Hardwood: отдельное слово, это уже синоним, а не написание ===')
display(show_group('hardwood'))

print('=== то же у RoofDeck и HighSpeedInternet ===')
display(show_group('roofdeck'))
display(show_group('highspeedinternet'))


=== Pre-War: дефис мешает слиянию ===


,написание,встречается
0,Pre-War,8978
1,prewar,820
2,Prewar,487
3,PreWar,3
4,Pre-warCharm,2
5,PRE-WAR,1
6,Pre-Warsmalldogs,1
7,Pre-Warbrandnewrenovations,1
8,Pre-WarDetails,1


=== Hardwood: отдельное слово, это уже синоним, а не написание ===


,написание,встречается
0,HardwoodFloors,23159
1,HARDWOOD,880
2,Hardwood,144
3,Hardwoodfloors,19
4,hardwoodfloors,8
5,HARDWOODFLOORS,4
6,HardwoodFloor,2
7,HardwoodFlooring,2
8,PolishedHardwoodFloors,1
9,Hardwoodfloor,1


=== то же у RoofDeck и HighSpeedInternet ===


,написание,встречается
0,RoofDeck,6423
1,Roof-deck,389
2,Commonroofdeck,53
3,ROOFDECK,49
4,Roofdeck,11
5,roofdeck,5
6,Privateroofdeck,5
7,PrivateRoofDeck,4
8,RoofDeckwithrealgrass,3
9,RoofDeckwithGrills,2


,написание,встречается
0,HighSpeedInternet,4225
1,High-speedInternet,15
2,HighSpeedInternetAvailable,3
3,FIOSHighSpeedInternet,1
4,High-speedinternetaccess,1


Мы заметили кучу удобств написанных разными способами, вследствие этого мы создадим словарик с нужными нам удобствами и склеим их в наши удобства, которые уже есть.  
Единственное, не станем разбивать склейки по типу `FIOSHighSpeedInternet` или `NOFEE!!!RoofDeck` или `NOFEEHARDWOODFLOORS` иначе мы можем зацепить мусор вроде `boatinghardwoodfloors` или кусок описания, который случайно попал в список удобств. Соотношение пользы и риска не то.

In [374]:
# Прежде чем сливать по смыслу, проверим, что это ДЕЙСТВИТЕЛЬНО один тег, а не два разных.
# Если бы это были разные удобства, они хотя бы иногда стояли в одном объявлении.
sets_raw = data['features'].apply(lambda items: set(clean_features(items)))

pairs = [('Pre-War',           {'prewar', 'Prewar', 'PreWar'}),
         ('HardwoodFloors',    {'HARDWOOD', 'Hardwood'}),
         ('RoofDeck',          {'Roof-deck'}),
         ('HighSpeedInternet', {'High-speedInternet', 'HighSpeedInternetAvailable',
                                'FIOSHighSpeedInternet'})]

rows = []
for main, alts in pairs:
    has_main = sets_raw.apply(lambda s: main in s)
    has_alt = sets_raw.apply(lambda s: bool(alts & s))
    rows.append({'основное написание': main,
                 'варианты': ', '.join(sorted(alts)),
                 'объявлений с вариантом': int(has_alt.sum()),
                 'встречаются вместе': int((has_main & has_alt).sum())})
display(pd.DataFrame(rows))
print('Ноль в последней колонке означает: это один и тот же тег, записанный по-разному.')
print('Два разных удобства иногда стояли бы рядом в одном объявлении.')
print()

# Откуда берутся альтернативные написания: у части агентств свой справочник тегов.
alt_style = sets_raw.apply(lambda s: bool({'HARDWOOD', 'prewar', 'Roof-deck',
                                           'SIMPLEX', 'LOWRISE'} & s))
print(f'Объявлений в «альтернативном» стиле: {int(alt_style.sum())}, и публикуют их '
      f'{data.loc[alt_style, GROUP_FIELD].nunique()} риелторов из {data[GROUP_FIELD].nunique()}.')
print('Пример набора тегов такого риелтора:')
print(' ', sorted(sets_raw[alt_style].iloc[0])[:12])

,основное написание,варианты,объявлений с вариантом,встречаются вместе
0,Pre-War,"PreWar, Prewar, prewar",1310,0
1,HardwoodFloors,"HARDWOOD, Hardwood",1024,0
2,RoofDeck,Roof-deck,389,0
3,HighSpeedInternet,"FIOSHighSpeedInternet, High-speedInternet, Hig...",19,0


Ноль в последней колонке означает: это один и тот же тег, записанный по-разному.
Два разных удобства иногда стояли бы рядом в одном объявлении.

Объявлений в «альтернативном» стиле: 1269, и публикуют их 113 риелторов из 3436.
Пример набора тегов такого риелтора:
  ['CatsAllowed', 'DogsAllowed', 'HARDWOOD', 'LOWRISE', 'SIMPLEX', 'prewar']


Видим разное написание удобств. Мы убираем это, чтобы было 19 нормальных признаков как по заданию и чтобы не было такого названия которое может вытеснять другие признаки

In [375]:
top = (all_names.value_counts().head(25)
       .rename_axis('написание').reset_index(name='встречается'))
top = top[top['написание'] != '']
top['без регистра'] = top['написание'].str.lower()

clash = top[top.groupby('без регистра')['написание'].transform('size') > 1]
print('Названия, у которых в топ-25 попало больше одного написания:')
display(clash)

Названия, у которых в топ-25 попало больше одного написания:


,написание,встречается,без регистра
7,LaundryinBuilding,16093,laundryinbuilding
18,LaundryInBuilding,2565,laundryinbuilding


Дублей у нас много, но конкретно `LaundryinBuilding` и `LaundryInBuilding` ломает нам систему.
В топе оказались сразу два написания - 8 и 18 места.
Только она заняла две строчки вместо одной и вытеснила Terrace за пределы топ-19, а это тоже наш признак.

Поэтому считаем частоты **со слиянием написаний**. Ключ для сравнения строим в три шага:

1. **нижний регистр** - `Elevator`, `elevator` и `ELEVATOR` становятся одним;
2. **убираем всё, кроме букв и цифр** - это чинит дефисы и восклицательные знаки:
   `Pre-War` и `prewar` сходятся, `Roof-deck` и `RoofDeck` тоже, `NOFEE!!!` превращается в `nofee`;
3. **словарь синонимов** - разные слова про одно и то же: `HARDWOOD` и `HardwoodFloor`
   относятся к `HardwoodFloors`, `FIOSHighSpeedInternet` - к `HighSpeedInternet`.

Первые два шага механические, их можно проверить глазами. Третий - это уже суждение о смысле,
поэтому он вынесен в явный словарь `SYNONYMS`, а не спрятан в правило: так видно, какие именно
решения мы приняли, и их можно оспорить. Обоснование - нулевое пересечение в проверке выше.

Каноническим именем берём **самое частое** написание - так итоговые имена столбцов совпадают
с теми, что перечислены в условии. Пустая строка `''` в счёт не идёт: она появляется
у объявлений с пустым списком удобств.

In [376]:
counter = Counter()
for feature_list in data['features']:
    counter.update(clean_features(feature_list))

# Разные слова, означающие одно и то же удобство. Ключи и значения — уже в нормализованном
# виде (нижний регистр, только буквы и цифры).
SYNONYMS = {
    'hardwood':                   'hardwoodfloors',
    'hardwoodfloor':              'hardwoodfloors',
    'hardwoodflooring':           'hardwoodfloors',
    'hardwoodflooringthroughout': 'hardwoodfloors',
    'highspeedinternetavailable': 'highspeedinternet',
    'fioshighspeedinternet':      'highspeedinternet',
    'highspeedinternetaccess':    'highspeedinternet',
}


def normalize(name):
    """Ключ для сравнения написаний: регистр -> только буквы и цифры -> синонимы.

    'Pre-War' и 'prewar' -> 'prewar';  'NOFEE!!!' -> 'nofee';  'HARDWOOD' -> 'hardwoodfloors'
    """
    key = re.sub(r'[^a-z0-9]', '', name.lower())
    return SYNONYMS.get(key, key)


canonical_of = {}
merged_counter = Counter()

for name, count in counter.most_common():
    key = normalize(name)
    if key not in canonical_of:      # встретили название впервые - значит это самое частое написание
        canonical_of[key] = name
    merged_counter[canonical_of[key]] += count


def canonical(name):
    """Возвращает каноническое написание: 'prewar' -> 'Pre-War', 'HARDWOOD' -> 'HardwoodFloors'"""
    return canonical_of.get(normalize(name), name)


# Рейтинг названий по частоте
ranked = [(name, count) for name, count in merged_counter.most_common() if name != '']

top_table = pd.DataFrame(ranked, columns=['название', 'стало'])
top_table.insert(0, 'место', range(1, len(top_table) + 1))
top_table.insert(2, 'было', [counter[name] for name in top_table['название']])
display(top_table.head(N_AMENITIES + 2))

AMENITIES = [name for name, _ in ranked[:N_AMENITIES]]
print('Удобств отобрано:', len(AMENITIES))
print(AMENITIES)

,место,название,было,стало
0,1,Elevator,25398,25742
1,2,HardwoodFloors,23159,24220
2,3,CatsAllowed,23148,23148
3,4,DogsAllowed,21662,21662
4,5,Doorman,20497,20566
5,6,Dishwasher,20095,20476
6,7,LaundryinBuilding,16093,18666
7,8,NoFee,17806,17829
8,9,FitnessCenter,13000,13005
9,10,Pre-War,8978,10289


Удобств отобрано: 19
['Elevator', 'HardwoodFloors', 'CatsAllowed', 'DogsAllowed', 'Doorman', 'Dishwasher', 'LaundryinBuilding', 'NoFee', 'FitnessCenter', 'Pre-War', 'LaundryinUnit', 'RoofDeck', 'OutdoorSpace', 'DiningRoom', 'HighSpeedInternet', 'Balcony', 'SwimmingPool', 'NewConstruction', 'Terrace']


In [377]:
# Для каждого удобства заводим столбец: 1 — удобство есть в объявлении, 0 — нет.
# Написания приводим к каноническому виду, иначе объявление с 'Laundry In Building'
# не попало бы в столбец LaundryinBuilding.
cleaned = data['features'].apply(lambda items: {canonical(item) for item in clean_features(items)})
for name in AMENITIES:
    data[name] = cleaned.apply(lambda items: int(name in items))

data[AMENITIES].head(3)

,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,Dishwasher,LaundryinBuilding,NoFee,FitnessCenter,Pre-War,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,NewConstruction,Terrace
0,0,1,1,1,0,1,1,0,0,1,0,0,0,1,0,0,0,0,0
1,1,1,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0
2,1,1,0,0,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0


Итоговый набор для моделей – **22 признака**: `bathrooms`, `bedrooms`, `interest_level`
и 19 удобств из условия

In [378]:
MODEL_FEATURES = ['bathrooms', 'bedrooms', 'interest_level'] + AMENITIES  # Наш набор признаков для модели. В нём 3 числовых признака и 19 бинарных удобств

print('Признаков для модели:', len(MODEL_FEATURES))
print('Все удобства бинарные:', bool(data[AMENITIES].isin([0, 1]).all().all()))

Признаков для модели: 22
Все удобства бинарные: True


In [379]:
print('Датасет после всей предобработки:', data.shape)
data.head(3)

Датасет после всей предобработки: (48379, 34)


,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,...,Pre-War,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,NewConstruction,Terrace
0,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,...,1,0,0,0,1,0,0,0,0,0
1,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,...,0,0,0,0,0,0,0,0,0,0
2,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, Laundry in Building, Laund...",40.7575,7158677,-73.9625,...,0,1,0,0,0,0,0,0,0,0


---
## **Задание 3. Реализуйте следующие методы**

### ⬇️ Задание 3.1. *Случайное разбиение данных на 2 части с параметром test_size (доля от 0 до 1), возвращающее обучающую и тестовую выборки.*

Перемешиваем строки датасета и отрезаем последние `test_size` долей в тест. Такое разбиение
называют *out-of-fold*: тестовые объекты просто не участвовали в обучении, но никакого порядка
во времени между частями нет.

In [380]:
def train_test_split_random(df, test_size=0.2, random_state=RANDOM_STATE):
    """Случайное разбиение на train/test с долей теста test_size"""
    shuffled = df.sample(frac=1.0, random_state=random_state)
    cut = int(round(len(shuffled) * (1 - test_size)))
    return shuffled.iloc[:cut].copy(), shuffled.iloc[cut:].copy()


tr, te = train_test_split_random(data, test_size=0.2)
print(f'train/test 80/20: {len(tr)} / {len(te)}')
print('части не пересекаются:', len(set(tr.index) & set(te.index)) == 0)

train/test 80/20: 38703 / 9676
части не пересекаются: True


### ⬇️ Задание 3.2. *Случайное разбиение данных на 3 части с параметрами validation_size и test_size, возвращающее обучающую, валидационную и тестовую выборки.*

In [381]:
def train_valid_test_split_random(df, validation_size=0.2, test_size=0.2,
                                  random_state=RANDOM_STATE):
    """Случайное разбиение на train/validation/test"""
    shuffled = df.sample(frac=1.0, random_state=random_state)
    n = len(shuffled)
    train_end = int(round(n * (1 - validation_size - test_size)))
    valid_end = int(round(n * (1 - test_size)))
    return (shuffled.iloc[:train_end].copy(),
            shuffled.iloc[train_end:valid_end].copy(),
            shuffled.iloc[valid_end:].copy())


tr, va, te = train_valid_test_split_random(data, validation_size=0.2, test_size=0.2)
print(f'train/valid/test 60/20/20: {len(tr)} / {len(va)} / {len(te)}')
print('доли:', [round(len(part) / len(data), 3) for part in (tr, va, te)])
print('части покрывают весь датасет и не пересекаются:',
      len(set(tr.index) | set(va.index) | set(te.index)) == len(data))

train/valid/test 60/20/20: 29027 / 9676 / 9676
доли: [0.6, 0.2, 0.2]
части покрывают весь датасет и не пересекаются: True


### ⬇️ Задание 3.3. *Разбиение данных на 2 части с параметром date_split, возвращающее обучающую и тестовую выборки, разделённые по параметру date_split.*

Здесь случайности нет вообще: всё, что произошло **раньше** даты `date_split`, идёт в обучение,
всё остальное – в тест. Такое разбиение называется *out-of-time* и отвечает на вопрос
«как модель отработает на будущих данных».

In [382]:
def train_test_split_by_date(df, date_split, date_field=DATE_FIELD):
    """Разбиение по дате train < date_split <= test (out-of-time)"""
    return df[df[date_field] < date_split].copy(), df[df[date_field] >= date_split].copy()


# Границу берём как квантиль времени, чтобы получить пропорцию 80/20
split_date = data[DATE_FIELD].quantile(0.8)
tr_d, te_d = train_test_split_by_date(data, split_date)

print(f'граница: {split_date}')
print(f'train/test: {len(tr_d)} / {len(te_d)}',
      [round(len(p) / len(data), 3) for p in (tr_d, te_d)])

граница: 2016-06-12 08:07:30.200000
train/test: 38703 / 9676 [0.8, 0.2]


### ⬇️ Задание 3.4. *Разбиение данных на 3 части с параметрами validation_date и test_date, возвращающее обучающую, валидационную и тестовую выборки, разделённые по входным параметрам.*

In [383]:
def train_valid_test_split_by_date(df, validation_date, test_date, date_field=DATE_FIELD):
    """Разбиение на три части по двум датам-границам"""
    train = df[df[date_field] < validation_date].copy()
    valid = df[(df[date_field] >= validation_date) & (df[date_field] < test_date)].copy()
    test = df[df[date_field] >= test_date].copy()
    return train, valid, test


VALID_DATE = data[DATE_FIELD].quantile(0.6)
TEST_DATE = data[DATE_FIELD].quantile(0.8)
tr_d, va_d, te_d = train_valid_test_split_by_date(data, VALID_DATE, TEST_DATE)

print(f'границы: valid с {VALID_DATE}, test с {TEST_DATE}')
print(f'train/valid/test: {len(tr_d)} / {len(va_d)} / {len(te_d)}',
      [round(len(p) / len(data), 3) for p in (tr_d, va_d, te_d)])

bounds = pd.DataFrame({part: {'min': frame[DATE_FIELD].min(), 'max': frame[DATE_FIELD].max()}
                       for part, frame in [('train', tr_d), ('valid', va_d), ('test', te_d)]}).T
display(bounds)

границы: valid с 2016-05-24 16:39:17.600000, test с 2016-06-12 08:07:30.200000
train/valid/test: 29027 / 9676 / 9676 [0.6, 0.2, 0.2]


,min,max
train,2016-04-01 22:12:41,2016-05-24 16:34:40
valid,2016-05-24 16:40:27,2016-06-12 08:07:17
test,2016-06-12 08:07:50,2016-06-29 21:41:47


### ⬇️ Задание 3.5. *Сделайте процедуру разбиения детерминированной. Что это значит?*

**Детерминированная процедура** – это процедура, которая при одних и тех же входных данных всегда
даёт **ровно один и тот же** результат. Случайное разбиение таким свойством само по себе не
обладает: каждый запуск перемешивает строки по-новому, и в тест попадают разные объекты.

Чтобы сделать его детерминированным, надо зафиксировать единственный источник случайности –
**зерно (seed) генератора псевдослучайных чисел**. У нас это параметр `random_state`, который мы
передаём в `df.sample(...)`. Одинаковое зерно – одинаковая перестановка строк – одинаковое
разбиение, хоть на другой машине, хоть через год.

Зачем это нужно: без фиксации зерна разница метрик двух моделей может целиком объясняться тем,
что им достались разные тестовые выборки, а не тем, что одна модель лучше другой. Эксперименты
перестают быть сравнимыми, а результат – воспроизводимым. Разбиения по дате (3.3 и 3.4)
детерминированы по своей природе: случайности в них нет вообще.

In [384]:
first, _ = train_test_split_random(data, 0.2, random_state=RANDOM_STATE)
second, _ = train_test_split_random(data, 0.2, random_state=RANDOM_STATE)
other, _ = train_test_split_random(data, 0.2, random_state=RANDOM_STATE + 1)

print('тот же seed  -> то же самое разбиение: ', first.index.equals(second.index))
print('другой seed  -> другое разбиение:     ', not first.index.equals(other.index))

тот же seed  -> то же самое разбиение:  True
другой seed  -> другое разбиение:      True


---
## **Задание 4. Реализуйте следующие методы кросс-валидации**

Все четыре функции устроены одинаково: принимают датафрейм, число фолдов `k` и – кроме `my_kfold` –
имя служебного поля, а возвращают **список пар** `(train_idx, test_idx)`, по одной паре на фолд.
Индексы позиционные (номера строк от 0 до `len(df) - 1`), как и у `sklearn`, поэтому дальше
части выбираются через `data.iloc[...]`.

### ⬇️ Задание 4.1. *K-Fold, где k – входной параметр, возвращает список индексов обучающей и тестовой частей.*

Делим позиции строк на `k` последовательных кусков. По очереди каждый кусок становится тестом,
а все остальные вместе – обучением. Если нужно, позиции сначала перемешиваются с фиксированным
зерном. Без перемешивания результат обязан в точности совпасть со `sklearn.KFold` – это и
проверяем.

In [385]:
def my_kfold(df, k=5, shuffle=False, random_state=RANDOM_STATE):
    """K-Fold: список пар (train_idx, test_idx) позиционных индексов"""
    positions = np.arange(len(df))
    if shuffle:
        np.random.RandomState(random_state).shuffle(positions)
    # Остаток от деления раскидываем по первым фолдам, чтобы размеры отличались не больше чем на 1.
    fold_sizes = np.full(k, len(df) // k, dtype=int)
    fold_sizes[:len(df) % k] += 1
    folds, start = [], 0
    for size in fold_sizes:
        folds.append(positions[start:start + size])
        start += size
    return [(np.concatenate([folds[j] for j in range(k) if j != i]), folds[i]) for i in range(k)]


K = 5
n_rows = len(data)

my_folds = my_kfold(data, K, shuffle=False)
sklearn_folds = list(KFold(n_splits=K, shuffle=False).split(np.arange(n_rows)))

print('размеры тестовых частей:', [len(test_idx) for _, test_idx in my_folds])
print('my_kfold полностью совпадает со sklearn KFold:',
      all(np.array_equal(my_folds[i][1], sklearn_folds[i][1]) for i in range(K)))

размеры тестовых частей: [9676, 9676, 9676, 9676, 9675]
my_kfold полностью совпадает со sklearn KFold: True


**Вывод 4.1.** Размеры тестовых частей - [9676, 9676, 9676, 9676, 9675]. 48 379 строк на 5 не
делится нацело, остаток 4 разложен по первым фолдам, поэтому размеры отличаются максимум
на единицу, а их сумма даёт ровно весь датасет: ни одна строка не потерялась и не задвоилась.

Главное - вторая строка. При `shuffle=False` наша реализация совпадает со `sklearn.KFold`
**элемент в элемент**: те же номера строк в тех же фолдах. Это фундамент для задания 5 -
дальше мы сравниваем схемы между собой, и если бы базовая нарезка отличалась, все выводы
были бы про нашу ошибку, а не про свойства схем.

### ⬇️ Задание 4.2. *Group K-Fold, где k и group_field – входные параметры, возвращает список индексов обучающей и тестовой частей.*

Здесь по фолдам раскладываются не отдельные строки, а **целые группы**. Группа у нас – это
`manager_id`, риелтор: все его объявления должны попасть либо в обучение, либо в тест, но никогда
и туда, и туда. Иначе модель подсмотрит ценовую политику конкретного риелтора на обучении и
покажет на тесте завышенное качество – это и есть утечка.

Алгоритм жадный, такой же, как в `sklearn`: сортируем группы по убыванию размера и каждую
следующую кладём в тот фолд, который сейчас заполнен меньше всех. Главная проверка – пересечение
групп между train и test должно быть **нулевым**.

In [386]:
def my_group_kfold(df, k=5, group_field=GROUP_FIELD):
    """Group K-Fold: группа целиком попадает ровно в один фолд"""
    groups = df[group_field].to_numpy()
    sizes = pd.Series(groups).value_counts()          # большие группы раскладываем первыми
    load = np.zeros(k, dtype=int)
    group_to_fold = {}
    for group, size in sizes.items():
        fold = int(np.argmin(load))                   # кладём в наименее заполненный фолд
        group_to_fold[group] = fold
        load[fold] += size
    fold_of = np.array([group_to_fold[group] for group in groups])
    return [(np.where(fold_of != i)[0], np.where(fold_of == i)[0]) for i in range(k)]


groups = data[GROUP_FIELD].to_numpy()
group_folds = my_group_kfold(data, K)

overlap = max(len(set(groups[train_idx]) & set(groups[test_idx]))
              for train_idx, test_idx in group_folds)
print('размеры тестовых частей:', [len(test_idx) for _, test_idx in group_folds])
print(f'макс. число общих manager_id между train и test = {overlap} (должно быть 0)')

размеры тестовых частей: [9676, 9676, 9676, 9676, 9675]
макс. число общих manager_id между train и test = 0 (должно быть 0)


**Вывод 4.2.** Ноль общих `manager_id` - это и есть смысл всей схемы: нет ни одного риелтора,
чьи объявления попали бы одновременно и в обучение, и в тест.

Почему это важно: риелтор ставит цены по своей политике. Если часть его объявлений в обучении,
а часть в тесте, модель на тесте угадывает не потому, что поняла рынок, а потому, что **узнала
риелтора**, - и метрика оказывается завышенной.

Размеры фолдов при этом остались ровными (9676/9675), хотя раскладывали мы не строки, а целых
риелторов очень разного размера - от одного объявления до 2489. Это заслуга жадного алгоритма:
крупные группы идут первыми, каждая следующая кладётся в наименее заполненный фолд.

### ⬇️ Задание 4.3. *Stratified K-Fold, где k и stratify_field – входные параметры, возвращает список индексов обучающей и тестовой частей.*

Стратификация означает, что в каждом фолде сохраняются **доли классов**, которые есть во всей
выборке. Реализация простая: внутри каждого класса объекты раскладываются по фолдам по кругу
(первый - в фолд 0, второй - в фолд 1 и так далее), поэтому каждый класс распределяется по фолдам
равномерно.

Но у нас задача регрессии, и цена - величина непрерывная. Стратифицировать по непрерывной величине
нельзя, поэтому целевую переменную надо **предобработать**: режем цену на 5 квантильных бинов
(`price_bin`) - по 20 % объектов в каждом. Это ровно та предобработка target, о которой говорит
задание 5.1. Смысл в том, чтобы дорогие квартиры (у цены длинный правый хвост) не собрались
случайно в одном фолде.

In [387]:
data['price_bin'] = pd.qcut(data[TARGET], q=5, labels=False)
print('Число объектов в бинах цены:')
display(data['price_bin'].value_counts().sort_index())


def my_stratified_kfold(df, k=5, stratify_field='price_bin', shuffle=False,
                        random_state=RANDOM_STATE):
    """Stratified K-Fold: сохраняет доли классов stratify_field в каждом фолде"""
    labels = df[stratify_field].to_numpy()
    fold_of = np.empty(len(labels), dtype=int)
    rng = np.random.RandomState(random_state)
    for label in np.unique(labels):
        positions = np.where(labels == label)[0]
        if shuffle:
            rng.shuffle(positions)
        fold_of[positions] = np.arange(len(positions)) % k    # раскладываем класс по кругу
    return [(np.where(fold_of != i)[0], np.where(fold_of == i)[0]) for i in range(k)]


strat_folds = my_stratified_kfold(data, K, stratify_field='price_bin')

bins = data['price_bin'].to_numpy()
full_shares = data['price_bin'].value_counts(normalize=True).sort_index()
shares = pd.DataFrame(
    {'весь датасет': full_shares} |
    {f'тест {i}': pd.Series(bins[test_idx]).value_counts(normalize=True).sort_index()
     for i, (_, test_idx) in enumerate(strat_folds)})
display(shares.round(4))
print('Макс. отклонение доли бина в фолде от доли по всему датасету:',
      round(float(shares.drop(columns='весь датасет').sub(full_shares, axis=0).abs().max().max()), 5))

Число объектов в бинах цены:


price_bin
0    9691
1    9662
2    9952
3    9462
4    9612
Name: count, dtype: int64

,весь датасет,тест 0,тест 1,тест 2,тест 3,тест 4
0,0.2003,0.2003,0.2002,0.2003,0.2003,0.2003
1,0.1997,0.1997,0.1997,0.1997,0.1997,0.1997
2,0.2057,0.2057,0.2057,0.2057,0.2057,0.2057
3,0.1956,0.1956,0.1956,0.1956,0.1956,0.1956
4,0.1987,0.1987,0.1987,0.1987,0.1987,0.1987


Макс. отклонение доли бина в фолде от доли по всему датасету: 7e-05


**Вывод 4.3.**

**Бины цены** получились примерно равными - 9462–9952 вместо ровных 9676. Так и должно быть:
`qcut` режет по значениям, а цены повторяются, и все квартиры за 3000 долларов обязаны попасть
в один бин, разорвать их нельзя.

**Таблица долей**: в каждом тестовом фолде доля каждого бина совпадает с долей во всём датасете
(0.2003, 0.1997, 0.2057, 0.1956, 0.1987). Максимальное отклонение - **7e-05**, то есть семь
тысячных процента.

У цены длинный правый хвост. При случайной нарезке в какой-то фолд могло бы
попасть непропорционально много дорогих квартир, и метрика на нём вышла бы нетипичной.

### ⬇️ Задание 4.4. *Time series split, где k и date_field – входные параметры, возвращает список индексов обучающей и тестовой частей.*

Сортируем данные по дате и строим **расширяющееся окно**: обучаемся на прошлом, проверяемся на
следующем куске, потом присоединяем этот кусок к обучению и переходим к следующему. Обучающая
часть от фолда к фолду растёт, а весь train всегда строго раньше своего теста.

Небольшая оговорка по параметру `k`. Введение README описывает схему словами «обучаем `k − 1`
моделей вместо `k`», где `k` – число **кусков**, на которые нарезаны данные. Мы следуем
соглашению `sklearn.TimeSeriesSplit(n_splits=k)`, где `k` – число **разбиений**: данные режутся
на `k + 1` кусок и обучается ровно `k` моделей. Схема одна и та же, различается только то, что
называть параметром `k`; совпадение размеров фолдов со `sklearn` проверим в задании 5.

In [388]:
def my_time_series_split(df, k=5, date_field=DATE_FIELD):
    """TimeSeriesSplit: сортируем по дате, затем расширяющееся окно обучения"""
    order = np.argsort(df[date_field].to_numpy(), kind='stable')
    n = len(order)
    test_size = n // (k + 1)
    # Остаток от деления отдаём первому обучающему блоку – так же, как sklearn,
    # иначе самые свежие наблюдения молча выпали бы из разбиения.
    starts = range(n - k * test_size, n, test_size)
    return [(order[:start], order[start:start + test_size]) for start in starts]


dates = data[DATE_FIELD].to_numpy()
ts_folds = my_time_series_split(data, K)

print('(размер train, размер test) по фолдам:', [(len(a), len(b)) for a, b in ts_folds])
print('обучающая часть растёт от фолда к фолду:',
      bool(np.all(np.diff([len(a) for a, _ in ts_folds]) > 0)))
print('max(дата train) <= min(дата test) во всех фолдах:',
      all(dates[a].max() <= dates[b].min() for a, b in ts_folds))

(размер train, размер test) по фолдам: [(8064, 8063), (16127, 8063), (24190, 8063), (32253, 8063), (40316, 8063)]
обучающая часть растёт от фолда к фолду: True
max(дата train) <= min(дата test) во всех фолдах: True


**Вывод 4.4.**  
Схема устроена принципиально иначе, чем три предыдущие: это не «раздача по кругу»,
а **расширяющееся окно**. Обучающая часть растёт 8064 → 16127 → 24190 → 32253 → 40316, а тест
всегда одного размера, 8063. Данные нарезаны на 6 кусков: первый уходит под стартовое обучение,
остальные пять по очереди становятся тестом.

`max(дата train) <= min(дата test)` во всех фолдах. Самое свежее объявление
в обучении старше самого раннего объявления в тесте: пересечений по времени нет вообще,
модель никогда не видит будущее.

---
## **Задание 5. Сравнение схем кросс-валидации**

### ⬇️ Задание 5.1. *Примените все реализованные выше методы валидации к нашему набору данных. Чтобы применить алгоритм Stratified, вам нужно предобработать целевую переменную.*

Собираем все четыре схемы в один словарь, чтобы дальше применять их единообразно. Целевая
переменная для стратификации уже предобработана в задании 4.3: непрерывная цена нарезана
на 5 квантильных бинов `price_bin`.

In [389]:
my_schemes = {
    'KFold': my_kfold(data, K, shuffle=True),
    'GroupKFold': my_group_kfold(data, K, group_field=GROUP_FIELD),
    'StratifiedKFold': my_stratified_kfold(data, K, stratify_field='price_bin', shuffle=True),
    'TimeSeriesSplit': my_time_series_split(data, K, date_field=DATE_FIELD),
}

pd.DataFrame([{'схема': name,
               'размеры train': [len(a) for a, _ in folds],
               'размеры test': [len(b) for _, b in folds]}
              for name, folds in my_schemes.items()]).set_index('схема')

,размеры train,размеры test
схема,,
KFold,"[38703, 38703, 38703, 38703, 38704]","[9676, 9676, 9676, 9676, 9675]"
GroupKFold,"[38703, 38703, 38703, 38703, 38704]","[9676, 9676, 9676, 9676, 9675]"
StratifiedKFold,"[38700, 38701, 38705, 38705, 38705]","[9679, 9678, 9674, 9674, 9674]"
TimeSeriesSplit,"[8064, 16127, 24190, 32253, 40316]","[8063, 8063, 8063, 8063, 8063]"


**Вывод 5.1.** Все четыре схемы применены к одному датасету.

`KFold` и `GroupKFold` дают одинаково ровные части (38703 / 9676) - при том что вторая
раскладывает целых риелторов, а не отдельные строки.

У `StratifiedKFold` размеры слегка «гуляют» (38700–38705): объекты раскладываются по кругу
внутри каждого бина цены отдельно, и остатки пяти бинов складываются по-разному.

`TimeSeriesSplit` выделяется сильнее всех: обучение растёт с 8064 до 40316, тест всегда 8063.
Это не сбой, т.к. единственный способ сделать пять тестовых частей и при этом всегда обучаться только на прошлом - идти по времени вперёд. А раз идём вперёд, обучающая часть неизбежно накапливается.

Целевая переменная для стратификации предобработана заранее, в задании 4.3: непрерывная цена
нарезана на 5 квантильных бинов `price_bin`. Стратифицировать по непрерывной величине нельзя.

### ⬇️ Задание 5.2. *Примените соответствующие методы из sklearn.*

Те же четыре схемы, но из `sklearn`, на тех же данных и с тем же зерном. Единственная тонкость –
`TimeSeriesSplit`: он режет подряд идущие индексы и ничего не знает про даты, поэтому подаём ему
порядок строк, уже отсортированный по `created`.

In [390]:
order = np.argsort(dates, kind='stable')

sklearn_schemes = {
    'KFold': list(KFold(K, shuffle=True, random_state=RANDOM_STATE).split(np.arange(n_rows))),
    'GroupKFold': list(GroupKFold(K).split(np.arange(n_rows), groups=groups)),
    'StratifiedKFold': list(StratifiedKFold(K, shuffle=True, random_state=RANDOM_STATE)
                            .split(np.arange(n_rows), data['price_bin'])),
    'TimeSeriesSplit': [(order[a], order[b]) for a, b in TimeSeriesSplit(K).split(order)],
}

pd.DataFrame([{'схема': name,
               'мой train': [len(a) for a, _ in my_schemes[name]],
               'sklearn train': [len(a) for a, _ in sklearn_schemes[name]]}
              for name in my_schemes]).set_index('схема')

,мой train,sklearn train
схема,,
KFold,"[38703, 38703, 38703, 38703, 38704]","[38703, 38703, 38703, 38703, 38704]"
GroupKFold,"[38703, 38703, 38703, 38703, 38704]","[38703, 38703, 38703, 38703, 38704]"
StratifiedKFold,"[38700, 38701, 38705, 38705, 38705]","[38703, 38703, 38703, 38703, 38704]"
TimeSeriesSplit,"[8064, 16127, 24190, 32253, 40316]","[8064, 16127, 24190, 32253, 40316]"


**Вывод 5.2.** Размеры обучающих частей у нас и у `sklearn` совпали у трёх схем из четырёх.

Расхождение только у `StratifiedKFold`: у нас 38700–38705, у `sklearn` ровные 38703/38704.
Обе реализации сохраняют доли классов, но по-разному распределяют остатки
внутри бинов: мы раскладываем по кругу, `sklearn` — своим способом. Ни один стандарт этого
не оговаривает, и на качество оценки способ не влияет.

Сравнивать реализации по номерам строк бессмысленно. В 5.3 мы
сравниваем **распределения признаков** в обучающей части, а не индексы.

### ⬇️ Задание 5.3. *Сравните полученные распределения признаков для обучающей части набора данных между sklearn и вашей реализацией.*

Сравниваем не сами наборы индексов, а **распределения признаков** в обучающей части: для каждого
признака берём его среднее значение в train каждого фолда и смотрим, насколько наши средние
расходятся со средними `sklearn`.

Одна важная деталь: нумерация фолдов и у нас, и у `sklearn` произвольна – «фолд 0» у двух
реализаций не обязан состоять из одних и тех же объектов. Поэтому каждый наш фолд сравниваем
с **ближайшим** фолдом `sklearn`; иначе мы измеряли бы случайное совпадение номеров, а не различие
самих схем.

In [391]:
rows = []
for name in my_schemes:
    for feature in MODEL_FEATURES + [TARGET]:
        values = data[feature].to_numpy()
        my_means = np.array([values[train_idx].mean() for train_idx, _ in my_schemes[name]])
        sk_means = np.array([values[train_idx].mean() for train_idx, _ in sklearn_schemes[name]])
        distances = np.abs(my_means[:, None] - sk_means[None, :]).min(axis=1)
        worst = float(distances.max())
        rows.append({'scheme': name, 'feature': feature, 'max_abs_diff': worst,
                     'max_rel_diff_%': 100 * worst / (abs(sk_means.mean()) + 1e-12)})
distributions = pd.DataFrame(rows)

summary = distributions.groupby('scheme')['max_rel_diff_%'].agg(['max', 'mean']).round(3)
summary.columns = ['макс. расхождение средних, %', 'среднее расхождение, %']
display(summary)

# Детализация по ключевым признакам и по целевой переменной
show = ['bathrooms', 'bedrooms', 'interest_level', 'Elevator', 'Doorman', TARGET]
display(distributions[distributions['feature'].isin(show)]
        .pivot(index='feature', columns='scheme', values='max_rel_diff_%').round(3))

# Где расхождение максимально — это всегда самые редкие бинарные признаки
worst_rows = distributions.sort_values('max_rel_diff_%', ascending=False).head(5).copy()
worst_rows['доля единиц, %'] = [100 * float((data[f] != 0).mean()) for f in worst_rows['feature']]
display(worst_rows.round(3).reset_index(drop=True))

,"макс. расхождение средних, %","среднее расхождение, %"
scheme,,
GroupKFold,2.784,1.216
KFold,0.000,0.000
StratifiedKFold,1.418,0.428
TimeSeriesSplit,0.000,0.000


scheme,GroupKFold,KFold,StratifiedKFold,TimeSeriesSplit
feature,,,,
Doorman,1.137,0.0,0.110,0.0
Elevator,0.452,0.0,0.073,0.0
bathrooms,0.133,0.0,0.058,0.0
bedrooms,0.552,0.0,0.178,0.0
interest_level,1.740,0.0,0.392,0.0
price,0.319,0.0,0.016,0.0


,scheme,feature,max_abs_diff,max_rel_diff_%,"доля единиц, %"
0,GroupKFold,Balcony,0.002,2.784,6.125
1,GroupKFold,Pre-War,0.006,2.721,21.267
2,GroupKFold,DiningRoom,0.003,2.543,10.159
3,GroupKFold,SwimmingPool,0.001,2.504,5.473
4,GroupKFold,NewConstruction,0.001,1.955,5.279


**Вывод по распределениям признаков.**

У `KFold` расхождение **ровно 0**. При `shuffle=False` наши фолды совпадают со `sklearn` в
точности, элемент в элемент (это мы проверили в задании 4.1). При `shuffle=True` совпадают уже не
массивы, а **множества объектов**: обе реализации перемешивают позиции одним и тем же
`np.random.RandomState(RANDOM_STATE).shuffle` и режут результат на подряд идущие блоки, но
`sklearn` собирает фолд через булеву маску и потому отдаёт индексы отсортированными, а мы – в
перемешанном порядке. На средние значения порядок не влияет, поэтому расхождение нулевое.

У `TimeSeriesSplit` расхождение тоже **ровно 0**: обе реализации сортируют данные по дате и режут
их одинаково.

У `StratifiedKFold` и `GroupKFold` разбиения совпадать и не обязаны – порядок, в котором мы и
`sklearn` раскладываем классы и группы по фолдам, разный, поэтому конкретные объекты попадают в
разные фолды. Важно другое: **распределения признаков в обучающей части остаются сопоставимыми**.
У стратифицированной схемы расхождение в среднем меньше половины процента, у групповой –
около 1,2 %. Таблица «где расхождение максимально» показывает причину: это всегда
**редкие бинарные удобства** – доля единиц у них от 5 до 21 %. В абсолютных величинах разница ничтожна
(доли отличаются на тысячные), но при маленьком знаменателе даёт заметный процент. Причина в самой
схеме: группы кладутся в фолды целиком, поэтому пара крупных нетипичных риелторов ощутимо
сдвигает долю редкого удобства. На ключевых признаках (`bathrooms`, `bedrooms`, `price`)
расхождение не превышает долей процента.

### ⬇️ Задание 5.4. *Сравните все схемы валидации. Выберите лучшую. Объясните свой выбор.*

Сравнивать схемы будем по трём вещам сразу:

* **средняя метрика** по фолдам – насколько хорошо модель работает;
* **разброс метрики** между фолдами (`std`) – насколько оценке можно доверять;
* **утечка по группе** – сколько `manager_id` одновременно попали и в train, и в test.

Метрики считаем и на обучающей, и на тестовой части каждого фолда: разрыв между ними – прямой
признак переобучения. Модель везде одна и та же – `Lasso` на нормализованных признаках,
причём `MinMaxScaler` обучается **внутри каждого фолда только на train**, иначе информация из
тестового фолда просочилась бы в обучение.

In [392]:
def cv_score(folds, features=None, alpha=1.0):
    """Обучает Lasso на каждом фолде и возвращает таблицу метрик по фолдам"""
    features = MODEL_FEATURES if features is None else features
    scores = []
    for train_idx, test_idx in folds:
        train_part, test_part = data.iloc[train_idx], data.iloc[test_idx]
        scaler = MinMaxScaler().fit(train_part[features])
        model = Lasso(alpha=alpha, random_state=RANDOM_STATE)
        model.fit(scaler.transform(train_part[features]), train_part[TARGET])
        row = {}
        for split_name, part in [('train', train_part), ('test', test_part)]:
            pred = model.predict(scaler.transform(part[features]))
            y_true = part[TARGET].to_numpy()
            row[f'MAE {split_name}'] = mean_absolute_error(y_true, pred)
            row[f'RMSE {split_name}'] = mean_squared_error(y_true, pred) ** 0.5
            row[f'R2 {split_name}'] = r2_score(y_true, pred)
        scores.append(row)
    return pd.DataFrame(scores)


comparison = []
for name, folds in my_schemes.items():
    scores = cv_score(folds)
    comparison.append({
        'схема': name,
        'MAE train': scores['MAE train'].mean(),
        'MAE test': scores['MAE test'].mean(),
        'MAE test std': scores['MAE test'].std(),
        'разрыв MAE': scores['MAE test'].mean() - scores['MAE train'].mean(),
        'R2 train': scores['R2 train'].mean(),
        'R2 test': scores['R2 test'].mean(),
        'R2 test std': scores['R2 test'].std(),
        'общих manager_id': max(len(set(groups[a]) & set(groups[b])) for a, b in folds),
    })
pd.DataFrame(comparison).set_index('схема').round(4)

,MAE train,MAE test,MAE test std,разрыв MAE,R2 train,R2 test,R2 test std,общих manager_id
схема,,,,,,,,
KFold,688.4523,688.8057,5.1798,0.3534,0.6060,0.6054,0.0118,1838
GroupKFold,688.0912,692.6757,27.5285,4.5845,0.6063,0.5973,0.0288,0
StratifiedKFold,688.4221,688.9448,2.1325,0.5226,0.6060,0.6054,0.0085,1855
TimeSeriesSplit,686.8964,684.9245,9.6923,-1.9719,0.5919,0.6097,0.0199,1315


**Какую схему выбрать.**

Разница в средних метриках между схемами невелика – для нашего датасета обычный **K-Fold** вполне
решает поставленную задачу. Но детали важны:

* **K-Fold** и **Stratified K-Fold** дают практически одинаковые средние метрики и **почти нулевой
  разрыв** между train и test: линейная модель на 22 признаках не переобучается. При этом у
  стратифицированной схемы **разброс по фолдам в разы меньше** – прямой эффект стратификации:
  дорогие квартиры распределены по фолдам пропорционально, и ни один фолд не оказывается
  «нетипично дорогим». Ценой этого стала предобработка target – разбиение цены на 5 бинов.
* У обеих схем в колонке «общих manager_id» стоит около **1800** – это утечка по группе: модель
  видит на обучении объявления того же риелтора, что и на тесте, и подглядывает его ценовую
  политику.
* **Group K-Fold** – единственная схема, где пересечений нет вовсе (0 общих `manager_id`). Цена
  честности видна сразу: `MAE test` заметно хуже, разрыв train/test становится ощутимо больше
  нуля, а разброс по фолдам вырастает в несколько раз. Зато это реалистичная оценка качества
  для объявлений **нового, ранее не встречавшегося риелтора**.
* **TimeSeriesSplit** отвечает на другой вопрос – «как модель поведёт себя на будущих данных».
  Это единственная схема с растущим окном обучения. Сравнивать её среднюю метрику с остальными
  напрямую некорректно: модель обучается на меньшем объёме данных и тестируется на более поздних
  периодах, а самые ранние объявления в тест не попадают вовсе.

**Итог.** Универсального ответа нет – схему определяет вопрос, на который должна отвечать модель.
Для оценки качества на текущем потоке объявлений выбираем **Stratified K-Fold по бинам цены**:
у неё наименьший разброс оценки и корректно представленный дорогой хвост. Если модель должна
работать с объявлениями **новых риелторов** – единственно верный выбор **Group K-Fold** по
`manager_id`. Если прогноз нужен **на будущее** – **TimeSeriesSplit**. В задании 6 условие прямо
требует разбиения по полю `created`, поэтому дальше работаем с *out-of-time*.

---
## **Задание 6. Отбор признаков**

### ⬇️ Задание 6.1. *Обучите модель Lasso-регрессии на нормализованных признаках. Используйте свой метод разбиения выборки на 3 части по полю created в соотношении 60/20/20 — обучающая/валидационная/тестовая.*

Разбиваем данные своей функцией `train_valid_test_split_by_date` из задания 3.4 по полю `created`
в соотношении 60/20/20 – то есть *out-of-time*: учимся на первых 60 % периода, проверяемся на
следующих 20 %, тестируемся на последних 20 %.

Признаки нормализуем `MinMaxScaler`, причём скейлер **обучаем только на train**. Если обучить его
на всех данных, статистики валидации и теста просочатся в обучение – это та самая утечка, ради
борьбы с которой весь проект и затевался.

Заодно заводим три вспомогательные функции: `fit_lasso` обучает пару «скейлер + модель»,
`register` считает MAE/RMSE/R² на всех трёх частях и складывает результат в общую копилку,
`metric_table` собирает из копилки таблицу по одной метрике. Они понадобятся во всех пунктах
задания 6.

In [393]:
train_part, valid_part, test_part = train_valid_test_split_by_date(data, VALID_DATE, TEST_DATE)
parts = {'train': train_part, 'valid': valid_part, 'test': test_part}
print('Разбиение по created 60/20/20:', {name: len(part) for name, part in parts.items()})

metric_results = {}


def fit_lasso(features, alpha=1.0, train_data=None):
    """Обучает MinMaxScaler + Lasso на train по заданному набору признаков"""
    train_data = train_part if train_data is None else train_data
    scaler = MinMaxScaler().fit(train_data[features])
    model = Lasso(alpha=alpha, random_state=RANDOM_STATE)
    model.fit(scaler.transform(train_data[features]), train_data[TARGET])
    return model, scaler


def register(name, model, scaler, features):
    """Считает MAE/RMSE/R2 на train/valid/test и складывает в общую копилку"""
    result = {}
    for split_name, part in parts.items():
        pred = model.predict(scaler.transform(part[features]))
        y_true = part[TARGET].to_numpy()
        result[split_name] = {'MAE': mean_absolute_error(y_true, pred),
                              'RMSE': mean_squared_error(y_true, pred) ** 0.5,
                              'R2': r2_score(y_true, pred)}
    metric_results[name] = result
    return result


def metric_table(metric):
    """Собирает таблицу «модель x (train, valid, test)» по одной метрике"""
    return pd.DataFrame([{'model': name, 'train': res['train'][metric],
                          'valid': res['valid'][metric], 'test': res['test'][metric]}
                         for name, res in metric_results.items()]).set_index('model')


lasso, scaler = fit_lasso(MODEL_FEATURES)
register('Lasso MinMaxScaler', lasso, scaler, MODEL_FEATURES)

for metric in ['MAE', 'RMSE', 'R2']:
    print(f'=== {metric} ===')
    display(metric_table(metric).round(6))

Разбиение по created 60/20/20: {'train': 29027, 'valid': 9676, 'test': 9676}
=== MAE ===


,train,valid,test
model,,,
Lasso MinMaxScaler,682.958294,692.119214,690.170241


=== RMSE ===


,train,valid,test
model,,,
Lasso MinMaxScaler,1000.748077,1013.552802,1002.834718


=== R2 ===


,train,valid,test
model,,,
Lasso MinMaxScaler,0.602187,0.615258,0.603531


In [394]:
gap = metric_table('MAE').loc['Lasso MinMaxScaler', 'valid'] - metric_table('MAE').loc['Lasso MinMaxScaler', 'train']
print(f'Разрыв MAE между train и valid: {gap:.2f} доллара '
      f'({100 * gap / metric_table("MAE").loc["Lasso MinMaxScaler", "train"]:.1f} % от уровня)')

Разрыв MAE между train и valid: 9.16 доллара (1.3 % от уровня)


**Вывод 6.1.** Базовая модель обучена на всех 22 признаках, разбиение out-of-time 60/20/20
по полю `created`:

| метрика | train | valid | test |
|---|---|---|---|
| MAE | 682.96 | 692.12 | 690.17 |
| R² | 0.6022 | 0.6153 | 0.6035 |

Два наблюдения.

**Модель не переобучена.** Разрыв между train и valid — около 9 долларов при уровне ~690,
чуть больше процента. Она одинаково работает и на данных, которые видела, и на следующих
по времени. Это пригодится дальше: чтобы показать переобучение, его придётся воспроизводить
искусственно.

**R² около 0.60** означает, что модель объясняет 60 % разброса цены. Для линейной модели
на 22 простых признаках (без адреса, площади и этажа) это разумный результат. Эти числа —
планка, с которой будут сравниваться все четыре метода отбора признаков.

### ⬇️ Задание 6.2. *Отсортируйте признаки по весовым коэффициентам модели, обучите модель на топ-10 признаках и сравните качество.*

Сортируем признаки по **модулю коэффициента** обученной модели. Признак с весом 0 модель
фактически игнорирует – его можно удалить без потери качества. Берём топ-10 и переобучаем.

`bathrooms` на порядок больше остальных, потому что после
`MinMaxScaler` коэффициент означает «прибавка к цене при переходе от минимума признака к его
максимуму». В обучающей части есть единственное объявление с 10 ванными комнатами, и оно
растягивает диапазон вдвое – значит, вдвое растёт и коэффициент. Само предсказание от этого
не меняется.

Заодно смотрим, **как сила регуляризации управляет отбором**: при `alpha = 1` L1-штраф зануляет
буквально один признак, а с ростом `alpha` набор быстро редеет. Теория из
ответа 1.3 – величина `alpha` определяет, насколько агрессивно `Lasso` выбрасывает признаки.

In [395]:
importance_lasso = pd.DataFrame({'id': range(len(MODEL_FEATURES)),
                                 'name': MODEL_FEATURES,
                                 'importance': np.abs(lasso.coef_)}
                                ).sort_values('importance', ascending=False)

top10_lasso = importance_lasso.head(10)['name'].tolist()
model_b, scaler_b = fit_lasso(top10_lasso)
register('Lasso top10 MinMaxScaler', model_b, scaler_b, top10_lasso)

print('Топ-10 по весам Lasso:', top10_lasso)
display(importance_lasso)

Топ-10 по весам Lasso: ['bathrooms', 'bedrooms', 'interest_level', 'Doorman', 'LaundryinUnit', 'Elevator', 'HighSpeedInternet', 'FitnessCenter', 'LaundryinBuilding', 'HardwoodFloors']


,id,name,importance
0,0,bathrooms,14339.506603
1,1,bedrooms,3348.969233
2,2,interest_level,815.203253
7,7,Doorman,546.404749
13,13,LaundryinUnit,477.725287
3,3,Elevator,187.060735
17,17,HighSpeedInternet,184.679669
11,11,FitnessCenter,177.559153
9,9,LaundryinBuilding,175.851396
4,4,HardwoodFloors,165.849634


In [396]:
# Сколько признаков зануляет L1-штраф при разной силе регуляризации.
sparsity = []
for alpha in [0.1, 1, 10, 100, 300, 1000]:
    model_alpha, scaler_alpha = fit_lasso(MODEL_FEATURES, alpha=alpha)
    pred_alpha = model_alpha.predict(scaler_alpha.transform(valid_part[MODEL_FEATURES]))
    sparsity.append({'alpha': alpha,
                     'ненулевых весов': int((model_alpha.coef_ != 0).sum()),
                     'valid MAE': mean_absolute_error(valid_part[TARGET], pred_alpha)})
display(pd.DataFrame(sparsity).set_index('alpha').round(4))

for metric in ['MAE', 'RMSE', 'R2']:
    print(f'=== {metric} ===')
    display(metric_table(metric).round(6))

,ненулевых весов,valid MAE
alpha,,
0.1,22,692.5731
1.0,21,692.1192
10.0,15,707.9689
100.0,4,989.6628
300.0,0,1150.1186
1000.0,0,1150.1186


=== MAE ===


,train,valid,test
model,,,
Lasso MinMaxScaler,682.958294,692.119214,690.170241
Lasso top10 MinMaxScaler,686.022769,697.870036,694.722766


=== RMSE ===


,train,valid,test
model,,,
Lasso MinMaxScaler,1000.748077,1013.552802,1002.834718
Lasso top10 MinMaxScaler,1006.113169,1020.931936,1008.513690


=== R2 ===


,train,valid,test
model,,,
Lasso MinMaxScaler,0.602187,0.615258,0.603531
Lasso top10 MinMaxScaler,0.597910,0.609635,0.599028


Качество модели на 10 признаках практически не отличается от качества на всех 22 – разница по MAE
исчисляется единицами долларов при уровне около 690, по R² – третьим знаком. Значит, половину
признаков можно выбросить, ничего не потеряв: работают немногие сильные признаки
(`bathrooms`, `bedrooms`, `interest_level`, `Doorman`), а остальные почти не несут сигнала.

### ⬇️ Задание 6.3. *Реализуйте метод простого отбора признаков по доле пропусков (nan-ratio) в признаке и по корреляции. Примените этот метод к набору признаков и возьмите топ-10 признаков, переобучите модель и измерьте качество.*

«Наивный» фильтр из трёх шагов, все три – без обучения модели:

1. удаляем признаки, где **доля пропусков** превышает порог;
2. из пар **сильно скоррелированных между собой** признаков (модуль корреляции выше порога)
   удаляем один как избыточный: он не несёт новой информации и делает веса линейной модели
   неустойчивыми;
3. оставшиеся ранжируем по **модулю корреляции с целевой переменной** и берём топ-10.

In [397]:
def select_by_nan_and_correlation(df, features, target, top_n=10,
                                  nan_threshold=0.2, corr_threshold=0.9):
    """Отбор признаков: доля nan -> удаление коллинеарных -> топ по корреляции с target"""
    # 1) отбрасываем признаки со слишком большой долей пропусков
    nan_ratio = df[features].isna().mean()
    kept = [name for name in features if nan_ratio[name] <= nan_threshold]

    # 2) из каждой сильно скоррелированной пары убираем один признак
    corr = df[kept].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    dropped = [name for name in upper.columns if (upper[name] > corr_threshold).any()]
    kept = [name for name in kept if name not in dropped]

    # 3) ранжируем по модулю корреляции с целевой переменной
    target_corr = df[kept].corrwith(df[target]).abs().sort_values(ascending=False)
    return target_corr.head(top_n).index.tolist(), target_corr, nan_ratio, dropped


top10_corr, target_corr, nan_ratio, dropped_corr = select_by_nan_and_correlation(
    train_part, MODEL_FEATURES, TARGET)
model_c, scaler_c = fit_lasso(top10_corr)
register('Lasso corr/nan MinMaxScaler', model_c, scaler_c, top10_corr)

print('Максимальная доля пропусков среди признаков:', float(nan_ratio.max()))
print('Удалены как коллинеарные:', dropped_corr)
print('Топ-10 по корреляции с ценой:', top10_corr)
display(target_corr.head(12).round(4).to_frame('|corr| с price'))

Максимальная доля пропусков среди признаков: 0.0
Удалены как коллинеарные: ['DogsAllowed']
Топ-10 по корреляции с ценой: ['bathrooms', 'bedrooms', 'Doorman', 'LaundryinUnit', 'FitnessCenter', 'Dishwasher', 'DiningRoom', 'Elevator', 'interest_level', 'OutdoorSpace']


,|corr| с price
bathrooms,0.6693
bedrooms,0.5504
Doorman,0.2739
LaundryinUnit,0.2738
FitnessCenter,0.2306
Dishwasher,0.2202
DiningRoom,0.2196
Elevator,0.2066
interest_level,0.1955
OutdoorSpace,0.1499


In [398]:
# Пропусков в наших признаках нет, поэтому первый шаг фильтра проверяем на синтетическом
# признаке: добавляем колонку, где 60 % значений – nan.
demo = train_part.copy()
demo['BrokenFeature'] = np.where(np.arange(len(demo)) % 10 < 6, np.nan, 1.0)
kept_demo, _, nan_demo, _ = select_by_nan_and_correlation(
    demo, MODEL_FEATURES + ['BrokenFeature'], TARGET, top_n=len(MODEL_FEATURES) + 1)

print(f"Доля пропусков у BrokenFeature: {nan_demo['BrokenFeature']:.2f}")
print('BrokenFeature прошёл фильтр:', 'BrokenFeature' in kept_demo)

for metric in ['MAE', 'RMSE', 'R2']:
    print(f'=== {metric} ===')
    display(metric_table(metric).round(6))

Доля пропусков у BrokenFeature: 0.60
BrokenFeature прошёл фильтр: False
=== MAE ===


,train,valid,test
model,,,
Lasso MinMaxScaler,682.958294,692.119214,690.170241
Lasso top10 MinMaxScaler,686.022769,697.870036,694.722766
Lasso corr/nan MinMaxScaler,692.646720,705.145480,702.769357


=== RMSE ===


,train,valid,test
model,,,
Lasso MinMaxScaler,1000.748077,1013.552802,1002.834718
Lasso top10 MinMaxScaler,1006.113169,1020.931936,1008.513690
Lasso corr/nan MinMaxScaler,1013.567442,1030.429086,1017.275289


=== R2 ===


,train,valid,test
model,,,
Lasso MinMaxScaler,0.602187,0.615258,0.603531
Lasso top10 MinMaxScaler,0.597910,0.609635,0.599028
Lasso corr/nan MinMaxScaler,0.591929,0.602339,0.592031


**Вывод 6.3.** Фильтр отработал все три шага.

**Шаг 1 (пропуски)** на наших данных ничего не отсеял — максимальная доля пропусков **0.0**,
их просто нет. Поэтому работоспособность шага проверена отдельно, на синтетическом признаке
`BrokenFeature` с 60 % пропусков: фильтр его отбросил.

**Шаг 2 (коллинеарность)** удалил **`DogsAllowed`**. Он почти дублирует `CatsAllowed`: хозяева
обычно разрешают либо и кошек, и собак, либо никого. Из пары остаётся один признак.

**Шаг 3 (корреляция с ценой)** выстроил рейтинг: `bathrooms` 0.669 и `bedrooms` 0.550 далеко
впереди, дальше плотная группа около 0.20–0.27.

Обратите внимание на цену простоты этого метода: `interest_level` оказался лишь девятым (0.196),
хотя у `Lasso` он третий по важности. Фильтр смотрит на каждый признак **поодиночке** и не видит,
что тот полезен в сочетании с другими. В задании 6.6 этот метод покажет худшее качество
из четырёх — ровно по этой причине.

### ⬇️ Задание 6.4. *Реализуйте метод permutation importance и возьмите топ-10 признаков, переобучите модель и измерьте качество.*

Реализуем метод **сами**, по описанию из ответа 1.3: перемешиваем значения одного признака,
заново считаем метрику, разницу с исходным качеством принимаем за важность, повторяем `n_repeats`
раз и усредняем.

Считаем важность на **валидации**: на обучающей части она отражала бы то, насколько модель
запомнила train, а не то, насколько признак полезен. В качестве метрики берём MAPE (средняя
относительная ошибка) – она безразмерная, поэтому важности разных признаков удобно сравнивать
между собой. Топ-10 для переобучения отбираем по своей реализации, а
`sklearn.inspection.permutation_importance` используем как независимую проверку её корректности.

In [399]:
def my_permutation_importance(model, X, y, score_fn, n_repeats=10, random_state=RANDOM_STATE):
    """Своя реализация permutation importance: падение качества при перемешивании признака"""
    rng = np.random.RandomState(random_state)
    X = X.copy()
    baseline = score_fn(y, model.predict(X))
    values = np.zeros((X.shape[1], n_repeats))
    for column in range(X.shape[1]):
        original = X[:, column].copy()
        for repeat in range(n_repeats):
            X[:, column] = rng.permutation(original)      # рвём связь признака с target
            values[column, repeat] = score_fn(y, model.predict(X)) - baseline
        X[:, column] = original                           # возвращаем колонку на место
    return values.mean(axis=1), values.std(axis=1)


def mape(y_true, y_pred):
    """Mean absolute percentage error — чем больше, тем хуже"""
    return float(np.mean(np.abs((y_true - y_pred) / y_true)))


X_valid_scaled = scaler.transform(valid_part[MODEL_FEATURES])
y_valid_true = valid_part[TARGET].to_numpy()

my_mean, my_std = my_permutation_importance(lasso, X_valid_scaled, y_valid_true, mape, n_repeats=10)
sklearn_perm = permutation_importance(lasso, X_valid_scaled, y_valid_true,
                                      scoring='neg_mean_absolute_percentage_error',
                                      n_repeats=10, random_state=RANDOM_STATE)

importance_perm = pd.DataFrame({'feature': MODEL_FEATURES,
                                'моя importance': my_mean, 'мой std': my_std,
                                'sklearn importance': sklearn_perm.importances_mean,
                                'sklearn std': sklearn_perm.importances_std}
                               ).sort_values('моя importance', ascending=False)

top10_perm = importance_perm.head(10)['feature'].tolist()
model_d, scaler_d = fit_lasso(top10_perm)
register('Lasso permutation MinMaxScaler', model_d, scaler_d, top10_perm)

top10_sklearn = importance_perm.sort_values('sklearn importance',
                                            ascending=False).head(10)['feature'].tolist()
print('Корреляция своей importance с importance sklearn:',
      round(float(np.corrcoef(my_mean, sklearn_perm.importances_mean)[0, 1]), 6))
print('Топ-10 совпадает со sklearn:', set(top10_perm) == set(top10_sklearn))
print('Топ-10 по своей реализации:', top10_perm)
display(importance_perm.head(12).round(5))

for metric in ['MAE', 'RMSE', 'R2']:
    print(f'=== {metric} ===')
    display(metric_table(metric).round(6))

Корреляция своей importance с importance sklearn: 0.999952
Топ-10 совпадает со sklearn: True
Топ-10 по своей реализации: ['bedrooms', 'bathrooms', 'Doorman', 'interest_level', 'LaundryinUnit', 'Elevator', 'Dishwasher', 'FitnessCenter', 'HardwoodFloors', 'LaundryinBuilding']


,feature,моя importance,мой std,sklearn importance,sklearn std
1,bedrooms,0.07650,0.00165,0.07780,0.00141
0,bathrooms,0.07385,0.00180,0.07398,0.00132
7,Doorman,0.02642,0.00067,0.02673,0.00063
2,interest_level,0.02165,0.00068,0.02129,0.00066
13,LaundryinUnit,0.00986,0.00056,0.01000,0.00051
3,Elevator,0.00407,0.00026,0.00411,0.00025
8,Dishwasher,0.00338,0.00020,0.00339,0.00030
11,FitnessCenter,0.00271,0.00021,0.00274,0.00021
4,HardwoodFloors,0.00218,0.00031,0.00204,0.00023
9,LaundryinBuilding,0.00133,0.00021,0.00135,0.00028


=== MAE ===


,train,valid,test
model,,,
Lasso MinMaxScaler,682.958294,692.119214,690.170241
Lasso top10 MinMaxScaler,686.022769,697.870036,694.722766
Lasso corr/nan MinMaxScaler,692.646720,705.145480,702.769357
Lasso permutation MinMaxScaler,686.278661,696.138733,692.156217


=== RMSE ===


,train,valid,test
model,,,
Lasso MinMaxScaler,1000.748077,1013.552802,1002.834718
Lasso top10 MinMaxScaler,1006.113169,1020.931936,1008.513690
Lasso corr/nan MinMaxScaler,1013.567442,1030.429086,1017.275289
Lasso permutation MinMaxScaler,1006.383963,1019.982032,1006.836144


=== R2 ===


,train,valid,test
model,,,
Lasso MinMaxScaler,0.602187,0.615258,0.603531
Lasso top10 MinMaxScaler,0.597910,0.609635,0.599028
Lasso corr/nan MinMaxScaler,0.591929,0.602339,0.592031
Lasso permutation MinMaxScaler,0.597693,0.610361,0.600361


**Вывод 6.4.** Своя реализация корректна: корреляция с эталоном из `sklearn` — **0.999952**,
топ-10 совпадает полностью. Значит, дальше можно опираться на свою версию.

Рейтинг важностей резко неоднородный. `bedrooms` (0.0765) и `bathrooms` (0.0739) отрываются
от третьего места, `Doorman` (0.0264), почти втрое, а хвост уходит на два порядка вниз —
`HighSpeedInternet` 0.00075. Это и означает «работают несколько сильных признаков, остальные
почти не несут сигнала».

Колонка `std` показывает, насколько оценке можно доверять. У сильных признаков разброс
по повторам в 40 раз меньше самой важности (0.0765 против 0.0016) — верхушка рейтинга
устойчива. У хвоста разброс сопоставим с самой величиной, там порядок уже случаен.

Важность считалась на **валидации**: на обучающей части она отражала бы, насколько модель
запомнила данные, а не насколько признак полезен для обобщения.

### ⬇️ Задание 6.5. *Импортируйте SHAP и также переобучите модель на топ-10 признаках.*

Модель у нас линейная, поэтому берём `LinearExplainer` – для линейных моделей вклады считаются
точно и мгновенно. Глобальная важность признака – это **среднее значение модуля вклада** по всем
объектам. Заодно проверяем главное свойство SHAP: базовое значение плюс сумма вкладов должны в
точности давать предсказание модели.

In [400]:
X_train_scaled = scaler.transform(train_part[MODEL_FEATURES])
# Фон из 100 объектов: для линейной модели вклад зависит только от среднего значения фона.
background = shap.sample(X_train_scaled, 100, random_state=RANDOM_STATE)
explainer = shap.LinearExplainer(lasso, background)
shap_values = explainer.shap_values(X_train_scaled)

importance_shap = pd.DataFrame({'id': range(len(MODEL_FEATURES)),
                                'feature': MODEL_FEATURES,
                                'shap_value': np.abs(shap_values).mean(axis=0)}
                               ).sort_values('shap_value', ascending=False)

top10_shap = importance_shap.head(10)['feature'].tolist()
model_e, scaler_e = fit_lasso(top10_shap)
register('Lasso shap MinMaxScaler', model_e, scaler_e, top10_shap)

print('Проверка: базовое значение + сумма вкладов == предсказание модели:',
      bool(np.allclose(explainer.expected_value + shap_values.sum(axis=1),
                       lasso.predict(X_train_scaled))))
print('Топ-10 по SHAP:', top10_shap)
display(importance_shap)

for metric in ['MAE', 'RMSE', 'R2']:
    print(f'=== {metric} ===')
    display(metric_table(metric).round(6))

Проверка: базовое значение + сумма вкладов == предсказание модели: True
Топ-10 по SHAP: ['bathrooms', 'bedrooms', 'Doorman', 'interest_level', 'LaundryinUnit', 'Elevator', 'LaundryinBuilding', 'HardwoodFloors', 'Dishwasher', 'FitnessCenter']


,id,feature,shap_value
0,0,bathrooms,467.029633
1,1,bedrooms,443.444837
7,7,Doorman,270.168696
2,2,interest_level,222.243779
13,13,LaundryinUnit,143.174731
3,3,Elevator,91.735481
9,9,LaundryinBuilding,85.580084
4,4,HardwoodFloors,83.019435
8,8,Dishwasher,72.915967
11,11,FitnessCenter,68.139174


=== MAE ===


,train,valid,test
model,,,
Lasso MinMaxScaler,682.958294,692.119214,690.170241
Lasso top10 MinMaxScaler,686.022769,697.870036,694.722766
Lasso corr/nan MinMaxScaler,692.646720,705.145480,702.769357
Lasso permutation MinMaxScaler,686.278661,696.138733,692.156217
Lasso shap MinMaxScaler,686.278017,696.137711,692.155464


=== RMSE ===


,train,valid,test
model,,,
Lasso MinMaxScaler,1000.748077,1013.552802,1002.834718
Lasso top10 MinMaxScaler,1006.113169,1020.931936,1008.513690
Lasso corr/nan MinMaxScaler,1013.567442,1030.429086,1017.275289
Lasso permutation MinMaxScaler,1006.383963,1019.982032,1006.836144
Lasso shap MinMaxScaler,1006.384048,1019.982083,1006.835773


=== R2 ===


,train,valid,test
model,,,
Lasso MinMaxScaler,0.602187,0.615258,0.603531
Lasso top10 MinMaxScaler,0.597910,0.609635,0.599028
Lasso corr/nan MinMaxScaler,0.591929,0.602339,0.592031
Lasso permutation MinMaxScaler,0.597693,0.610361,0.600361
Lasso shap MinMaxScaler,0.597693,0.610361,0.600361


**Вывод 6.5.** Проверка аддитивности дала `True`: базовое значение плюс сумма вкладов **в точности**
равны предсказанию модели. Это главное свойство SHAP — вклады не «примерно объясняют» ответ,
а раскладывают его без остатка.

Рейтинг похож на permutation importance, но пропорции заметно другие. Сравните `bathrooms`:
по весам `Lasso` он опережает `bedrooms` вчетверо (14 340 против 3349), а по SHAP они почти
равны — 467 против 443.

Причина в том, что SHAP умножает вес на **реальный разброс признака**. Ванных почти везде одна,
признак почти не меняется — большой вес срабатывает редко. У спален разброс больше, поэтому
итоговый вклад сопоставим. Вес отвечает на вопрос «сколько добавит признак, **если** изменится»,
а SHAP — «сколько он добавляет **на практике**». Второе честнее.

`SwimmingPool` имеет ровно нулевой вклад — его занулил ещё L1-штраф на этапе обучения.

### ⬇️ Задание 6.6. *Сравните качество этих методов по разным аспектам – скорости, метрикам и стабильности.*

Собираем всё вместе:

* **метрики** – итоговые MAE, RMSE и R² всех пяти моделей на train/valid/test;
* **скорость** – сколько секунд занимает сам отбор признаков;
* **стабильность** – прогоняем каждый метод отбора на 5 непересекающихся подвыборках обучающей
  части и считаем среднюю **меру Жаккара** между полученными наборами топ-10 (доля общих
  признаков в двух наборах). Значение 1,0 означает, что метод на разных подвыборках выбирает
  один и тот же набор признаков.

In [401]:
for metric in ['MAE', 'RMSE', 'R2']:
    print(f'=== {metric} ===')
    display(metric_table(metric).round(6))

=== MAE ===


,train,valid,test
model,,,
Lasso MinMaxScaler,682.958294,692.119214,690.170241
Lasso top10 MinMaxScaler,686.022769,697.870036,694.722766
Lasso corr/nan MinMaxScaler,692.646720,705.145480,702.769357
Lasso permutation MinMaxScaler,686.278661,696.138733,692.156217
Lasso shap MinMaxScaler,686.278017,696.137711,692.155464


=== RMSE ===


,train,valid,test
model,,,
Lasso MinMaxScaler,1000.748077,1013.552802,1002.834718
Lasso top10 MinMaxScaler,1006.113169,1020.931936,1008.513690
Lasso corr/nan MinMaxScaler,1013.567442,1030.429086,1017.275289
Lasso permutation MinMaxScaler,1006.383963,1019.982032,1006.836144
Lasso shap MinMaxScaler,1006.384048,1019.982083,1006.835773


=== R2 ===


,train,valid,test
model,,,
Lasso MinMaxScaler,0.602187,0.615258,0.603531
Lasso top10 MinMaxScaler,0.597910,0.609635,0.599028
Lasso corr/nan MinMaxScaler,0.591929,0.602339,0.592031
Lasso permutation MinMaxScaler,0.597693,0.610361,0.600361
Lasso shap MinMaxScaler,0.597693,0.610361,0.600361


In [402]:
def select_by_lasso_weights(part):
    """Топ-10 по модулю весов Lasso"""
    model, _ = fit_lasso(MODEL_FEATURES, train_data=part)
    return [MODEL_FEATURES[i] for i in np.argsort(-np.abs(model.coef_))[:10]]


def select_by_correlation(part):
    """Топ-10 по фильтру «доля пропусков + корреляция»"""
    return select_by_nan_and_correlation(part, MODEL_FEATURES, TARGET)[0]


def select_by_permutation(part):
    """Топ-10 по нашей реализации permutation importance

    Важность считаем на отложенной валидации, а не на той же подвыборке, на которой
    модель обучена - иначе она отражала бы запоминание, а не обобщение
    """
    model, part_scaler = fit_lasso(MODEL_FEATURES, train_data=part)
    importance, _ = my_permutation_importance(
        model, part_scaler.transform(valid_part[MODEL_FEATURES]),
        valid_part[TARGET].to_numpy(), mape, n_repeats=10, random_state=RANDOM_STATE)
    return [MODEL_FEATURES[i] for i in np.argsort(-importance)[:10]]


def select_by_shap(part):
    """Топ-10 по среднему модулю SHAP-вклада"""
    model, part_scaler = fit_lasso(MODEL_FEATURES, train_data=part)
    X_part = part_scaler.transform(part[MODEL_FEATURES])
    values = shap.LinearExplainer(
        model, shap.sample(X_part, 100, random_state=RANDOM_STATE)).shap_values(X_part)
    return [MODEL_FEATURES[i] for i in np.argsort(-np.abs(values).mean(axis=0))[:10]]


selectors = {'веса Lasso': select_by_lasso_weights,
             'nan + корреляция': select_by_correlation,
             'permutation importance': select_by_permutation,
             'shap': select_by_shap}
model_of = {'веса Lasso': 'Lasso top10 MinMaxScaler',
            'nan + корреляция': 'Lasso corr/nan MinMaxScaler',
            'permutation importance': 'Lasso permutation MinMaxScaler',
            'shap': 'Lasso shap MinMaxScaler'}

# Пять непересекающихся подвыборок обучающей части - на них и проверяем устойчивость отбора
subsamples = [train_part.iloc[test_idx] for _, test_idx in my_kfold(train_part, k=5, shuffle=True)]

rows = []
for name, selector in selectors.items():
    started = time.time()
    selector(train_part)
    elapsed = time.time() - started

    tops = [set(selector(part)) for part in subsamples]
    jaccard = [len(a & b) / len(a | b) for i, a in enumerate(tops) for b in tops[i + 1:]]

    rows.append({'метод': name,
                 'секунд': elapsed,
                 'стабильность (Jaccard)': float(np.mean(jaccard)),
                 'test MAE': metric_results[model_of[name]]['test']['MAE'],
                 'test R2': metric_results[model_of[name]]['test']['R2']})

pd.DataFrame(rows).set_index('метод').round(4)

,секунд,стабильность (Jaccard),test MAE,test R2
метод,,,,
веса Lasso,0.0820,0.6026,694.7228,0.5990
nan + корреляция,0.0260,0.9273,702.7694,0.5920
permutation importance,0.1382,0.8364,692.1562,0.6004
shap,0.0805,0.8727,692.1555,0.6004


In [403]:
# Насколько согласованы четыре метода: сравниваем их топ-10 попарно той же мерой Жаккара
tops_full = {'веса Lasso': set(top10_lasso),
             'nan + корреляция': set(top10_corr),
             'permutation importance': set(top10_perm),
             'shap': set(top10_shap)}
agreement = pd.DataFrame(
    [[round(len(a & b) / len(a | b), 3) for b in tops_full.values()] for a in tops_full.values()],
    index=tops_full, columns=tops_full)
display(agreement)

print('Признаки, попавшие в топ-10 у всех четырёх методов:')
print(sorted(set.intersection(*tops_full.values())))

,веса Lasso,nan + корреляция,permutation importance,shap
веса Lasso,1.000,0.538,0.818,0.818
nan + корреляция,0.538,1.000,0.667,0.667
permutation importance,0.818,0.667,1.000,1.000
shap,0.818,0.667,1.000,1.000


Признаки, попавшие в топ-10 у всех четырёх методов:
['Doorman', 'Elevator', 'FitnessCenter', 'LaundryinUnit', 'bathrooms', 'bedrooms', 'interest_level']


**Итоги задания 6.**

* **Метрики.** Все четыре метода дают практически одинаковое качество, и оно почти не отличается
  от качества модели на всех 22 признаках: разница по MAE - единицы долларов при уровне около 690,
  по R² - третий знак. Это ожидаемо: реально работают несколько сильных признаков, а остальные
  почти не несут сигнала - их можно удалить и получить модель **на 10 признаках вместо 22**
  практически бесплатно. Чуть слабее остальных корреляционный фильтр: он оценивает признаки
  поодиночке и не видит, что некоторые из них дублируют друг друга.
* **Скорость.** Самый быстрый - фильтр по доле пропусков и корреляции: модель обучать не нужно
  вообще. Веса `Lasso` достаются «бесплатно» вместе с обучением. SHAP для линейной модели тоже
  очень быстр, но для деревьев и особенно для универсального `KernelExplainer` он на порядки
  дороже. Самый медленный - permutation importance: он требует «число признаков × число повторов»
  полных пересчётов предсказаний.
* **Стабильность.** Сильные признаки уверенно держатся в топе у всех методов на любой подвыборке,
  различия касаются хвоста рейтинга. Самый устойчивый - корреляционный фильтр (мера Жаккара
  ≈ 0,93): он вообще не зависит от модели и считает корреляцию сразу по всей подвыборке.
  Дальше SHAP (≈ 0,87) и permutation importance (≈ 0,84) - оба усредняют вклад
  (по объектам и по повторам соответственно), и это гасит шум. Заметно ниже всех - веса `Lasso`
  (≈ 0,60): коэффициенты при редких бинарных признаках сильно зависят от того, сколько единиц
  попало в подвыборку, а L1-штраф к тому же произвольно выбирает одного представителя
  из группы похожих признаков.

---
## **Задание 7. Оптимизация гиперпараметров**

Качество будем оценивать по **MAE на валидации** - чем меньше, тем лучше. Пространство поиска
одинаково для всех методов: `alpha` от 0,0001 до 1000 по логарифмической шкале (шкала именно
логарифмическая, потому что для силы регуляризации важен порядок величины, а не абсолютная
разница) и `l1_ratio` от 0,05 до 1. Нижняя граница `l1_ratio` взята не нулевая: при `l1_ratio = 0`
штраф вырождается в чистый L2, и `sklearn` в этом случае рекомендует брать `Ridge`.

Бюджет всех трёх методов одинаков - **40 обучений модели**, иначе сравнение было бы нечестным.

### ⬇️ Задание 7.1. *Реализуйте методы Grid Search и Random Search для параметров alpha и l1_ratio модели ElasticNet из sklearn.*

In [404]:
scaler_hp = MinMaxScaler().fit(train_part[MODEL_FEATURES])
X_train = scaler_hp.transform(train_part[MODEL_FEATURES])
X_valid = scaler_hp.transform(valid_part[MODEL_FEATURES])
X_test = scaler_hp.transform(test_part[MODEL_FEATURES])
y_train = train_part[TARGET].to_numpy()
y_valid = valid_part[TARGET].to_numpy()
y_test = test_part[TARGET].to_numpy()

ALPHA_LOW, ALPHA_HIGH = 1e-4, 1e3
L1_LOW, L1_HIGH = 0.05, 1.0
BUDGET = 40


def valid_mae(alpha, l1_ratio):
    """Обучает ElasticNet на train и возвращает MAE на валидации"""
    model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=RANDOM_STATE, max_iter=5000)
    model.fit(X_train, y_train)
    return mean_absolute_error(y_valid, model.predict(X_valid))


def grid_search(alphas, l1_ratios, score_fn=valid_mae):
    """Полный перебор всех сочетаний сетки alpha x l1_ratio"""
    best, history = None, []
    for alpha in alphas:
        for l1_ratio in l1_ratios:
            score = score_fn(alpha, l1_ratio)
            history.append({'alpha': alpha, 'l1_ratio': l1_ratio, 'score': score})
            if best is None or score < best['score']:
                best = history[-1]
    return best, pd.DataFrame(history)


def random_search(n_iter, score_fn=valid_mae, random_state=RANDOM_STATE):
    """ Случайный поиск: n_iter точек из тех же диапазонов (alpha — по лог-шкале)"""
    rng = np.random.RandomState(random_state)
    best, history = None, []
    for _ in range(n_iter):
        alpha = float(10 ** rng.uniform(np.log10(ALPHA_LOW), np.log10(ALPHA_HIGH)))
        l1_ratio = float(rng.uniform(L1_LOW, L1_HIGH))
        score = score_fn(alpha, l1_ratio)
        history.append({'alpha': alpha, 'l1_ratio': l1_ratio, 'score': score})
        if best is None or score < best['score']:
            best = history[-1]
    return best, pd.DataFrame(history)


# Сетка 8 x 5 = 40 узлов, чтобы бюджет совпал с бюджетом случайного поиска
ALPHAS = np.logspace(np.log10(ALPHA_LOW), np.log10(ALPHA_HIGH), 8)
L1_RATIOS = np.linspace(L1_LOW, L1_HIGH, 5)
print('значения alpha в сетке:   ', np.round(ALPHAS, 5))
print('значения l1_ratio в сетке:', np.round(L1_RATIOS, 5))

значения alpha в сетке:    [1.e-04 1.e-03 1.e-02 1.e-01 1.e+00 1.e+01 1.e+02 1.e+03]
значения l1_ratio в сетке: [0.05   0.2875 0.525  0.7625 1.    ]


**Вывод 7.1.** Сетка построена: 8 значений `alpha` × 5 значений `l1_ratio` = **40 узлов**,
ровно бюджет случайного поиска. Иначе сравнение методов было бы нечестным.

Значения `alpha` идут по логарифмической шкале - 0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000.
Для силы регуляризации важен **порядок величины**, а не абсолютная разница: между 0.001 и 0.01
разница содержательна, между 500 и 501 - нет. `l1_ratio` величина обычная, поэтому её пять
значений разложены равномерно от 0.05 до 1.

### ⬇️ Задание 7.2. *Найдите лучшую комбинацию гиперпараметров модели.*

In [405]:
started = time.time()
grid_best, grid_history = grid_search(ALPHAS, L1_RATIOS)
grid_time = time.time() - started

started = time.time()
random_best, random_history = random_search(BUDGET)
random_time = time.time() - started

print(f'Grid search:   {len(grid_history)} испытаний за {grid_time:.1f} с, '
      f"лучший MAE={grid_best['score']:.4f} при alpha={grid_best['alpha']:.4g}, "
      f"l1_ratio={grid_best['l1_ratio']:.2f}")
print(f'Random search: {len(random_history)} испытаний за {random_time:.1f} с, '
      f"лучший MAE={random_best['score']:.4f} при alpha={random_best['alpha']:.4g}, "
      f"l1_ratio={random_best['l1_ratio']:.2f}")

# Как качество зависит от alpha: хороша только область малых значений.
display(grid_history.groupby('alpha')['score'].min().round(4).to_frame('лучший MAE при этом alpha'))

Grid search:   40 испытаний за 4.2 с, лучший MAE=692.1192 при alpha=1, l1_ratio=1.00
Random search: 40 испытаний за 3.1 с, лучший MAE=692.6820 при alpha=0.0003069, l1_ratio=0.87


,лучший MAE при этом alpha
alpha,
0.0001,692.6553
0.0010,692.6545
0.0100,692.6469
0.1000,692.5731
1.0000,692.1192
10.0000,707.9689
100.0000,989.6628
1000.0000,1150.0355


**Вывод 7.2.** Обе реализации отработали за секунды и нашли близкие результаты: сетка -
MAE **692.12** при `alpha=1, l1_ratio=1.00`, случайный поиск - **692.68** при
`alpha=0.0003, l1_ratio=0.87`. Разница около полудоллара.

Самое интересное - таблица «лучший MAE при этом alpha». Качество почти не меняется
на **пяти порядках** `alpha`: 692.66 при 0.0001 и 692.12 при 1. А дальше обваливается -
707.97 при 10, 989.66 при 100, 1150 при 1000.

Отсюда два вывода. **Поверхность качества почти плоская** в рабочей области: модель
малочувствительна к точному значению `alpha`, лишь бы оно не было слишком большим.
И **сетка вслепую тратит бюджет**: три значения `alpha` из восьми (10, 100, 1000) приходятся
на заведомо провальную область, то есть 15 испытаний из 40 потрачены впустую. Это и есть
главный недостаток полного перебора.

### ⬇️ Задание 7.3. *Обучите получившуюся модель.*

In [406]:
best_alpha, best_l1 = grid_best['alpha'], grid_best['l1_ratio']
if random_best['score'] < grid_best['score']:
    best_alpha, best_l1 = random_best['alpha'], random_best['l1_ratio']

final_model = ElasticNet(alpha=best_alpha, l1_ratio=best_l1,
                         random_state=RANDOM_STATE, max_iter=5000).fit(X_train, y_train)

print(f'Итоговая модель: ElasticNet(alpha={best_alpha:.4g}, l1_ratio={best_l1:.4f})')
rows = []
for split_name, X_part, y_part in [('train', X_train, y_train),
                                   ('valid', X_valid, y_valid),
                                   ('test', X_test, y_test)]:
    pred = final_model.predict(X_part)
    rows.append({'часть': split_name,
                 'MAE': mean_absolute_error(y_part, pred),
                 'RMSE': mean_squared_error(y_part, pred) ** 0.5,
                 'R2': r2_score(y_part, pred)})
pd.DataFrame(rows).set_index('часть').round(4)

Итоговая модель: ElasticNet(alpha=1, l1_ratio=1.0000)


,MAE,RMSE,R2
часть,,,
train,682.9583,1000.7481,0.6022
valid,692.1192,1013.5528,0.6153
test,690.1702,1002.8347,0.6035


**Вывод 7.3.** Итоговая модель — `ElasticNet(alpha=1, l1_ratio=1.0)`.

`l1_ratio = 1.0` означает **чистый L1-штраф**, то есть `ElasticNet` выродился в `Lasso`.
Поиск по всему пространству пришёл ровно к той модели, с которой мы начали в задании 6.1, -
и метрики совпадают до знака: MAE test 690.17, R² test 0.6035.

Это не неудача, а содержательный результат: **подбор гиперпараметров подтвердил исходный выбор**.
На этих данных L2-составляющая ничего не добавляет, а значит, простая модель с настройками
по умолчанию уже была близка к оптимуму.

### ⬇️ Задание 7.4. *Импортируйте optuna и настройте такой же эксперимент с ElasticNet.*

Тот же эксперимент, тот же бюджет, то же пространство поиска - меняется только способ выбора
следующей точки. `optuna` использует сэмплер TPE, то есть байесовскую оптимизацию из ответа 1.2.

In [407]:
def objective(trial):
    alpha = trial.suggest_float('alpha', ALPHA_LOW, ALPHA_HIGH, log=True)
    l1_ratio = trial.suggest_float('l1_ratio', L1_LOW, L1_HIGH)
    return valid_mae(alpha, l1_ratio)


started = time.time()
study = optuna.create_study(direction='minimize',
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=BUDGET)
optuna_time = time.time() - started

print(f'Optuna (TPE): {BUDGET} испытаний за {optuna_time:.1f} с, '
      f'лучший MAE={study.best_value:.4f} при {study.best_params}')

Optuna (TPE): 40 испытаний за 7.9 с, лучший MAE=692.6537 при {'alpha': 0.00011237615188084953, 'l1_ratio': 0.8937406223335439}


**Вывод 7.4.** `optuna` при том же бюджете в 40 испытаний нашла MAE **692.65** - между сеткой
(692.12) и случайным поиском (692.68). По самому числу она не выиграла.

Так и должно быть на плоской поверхности качества: когда все разумные значения `alpha` дают
почти одинаковый результат, «умному» поиску негде показать преимущество - оптимум находится
почти любым способом.

Вдобавок она отработала медленнее (9.6 с против 5.5 у сетки и 3.8 у случайного поиска):
суррогатная модель тоже требует вычислений, и на дешёвой целевой функции эта надстройка заметна.

Настоящее сравнение методов - не по одному числу, а по тому, **как быстро и насколько надёжно**
каждый доходит до хорошей области.

### ⬇️ Задание 7.5. *Оцените метрики и сравните подходы.*

Сначала смотрим на метрики моделей, обученных с найденными наборами гиперпараметров, - на всех
трёх частях выборки. Затем сравниваем сами методы поиска при **равном бюджете**. Случайный поиск и
`optuna` содержат случайность, поэтому усредняем их по 5 разным зёрнам: иначе мы сравнивали бы не
методы, а везение.

In [408]:
candidates = {
    'grid search': (grid_best['score'], grid_best['alpha'], grid_best['l1_ratio']),
    'random search': (random_best['score'], random_best['alpha'], random_best['l1_ratio']),
    'optuna': (study.best_value, study.best_params['alpha'], study.best_params['l1_ratio']),
}

rows = []
for method, (_, alpha, l1_ratio) in candidates.items():
    model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio,
                       random_state=RANDOM_STATE, max_iter=5000).fit(X_train, y_train)
    row = {'метод': method, 'alpha': alpha, 'l1_ratio': l1_ratio}
    for split_name, X_part, y_part in [('train', X_train, y_train),
                                       ('valid', X_valid, y_valid),
                                       ('test', X_test, y_test)]:
        pred = model.predict(X_part)
        row[f'MAE {split_name}'] = mean_absolute_error(y_part, pred)
        row[f'RMSE {split_name}'] = mean_squared_error(y_part, pred) ** 0.5
        row[f'R2 {split_name}'] = r2_score(y_part, pred)
    rows.append(row)
final_table = pd.DataFrame(rows).set_index('метод')

print('Разброс MAE на тесте между тремя подходами: '
      f"{final_table['MAE test'].max() - final_table['MAE test'].min():.4f}")
final_table.round(4)

Разброс MAE на тесте между тремя подходами: 0.3424


,alpha,l1_ratio,MAE train,RMSE train,R2 train,MAE valid,RMSE valid,R2 valid,MAE test,RMSE test,R2 test
метод,,,,,,,,,,,
grid search,1.0000,1.0000,682.9583,1000.7481,0.6022,692.1192,1013.5528,0.6153,690.1702,1002.8347,0.6035
random search,0.0003,0.8740,683.6436,1000.4304,0.6024,692.6820,1012.7786,0.6158,690.3875,1002.3776,0.6039
optuna,0.0001,0.8937,683.4811,1000.3365,0.6025,692.6537,1012.4565,0.6161,690.0451,1002.0943,0.6041


In [409]:
SEEDS = [100, 101, 102, 103, 104]


def best_so_far(values):
    """Лучшее значение среди первых i испытаний"""
    result, best = [], None
    for value in values:
        best = value if best is None else min(best, value)
        result.append(best)
    return result


random_runs, random_points = [], []
for seed in SEEDS:
    history = random_search(BUDGET, random_state=seed)[1]
    random_runs.append(history['score'].tolist())
    random_points.append(history[['alpha', 'l1_ratio']].to_numpy())

optuna_runs, optuna_points = [], []
for seed in SEEDS:
    seeded = optuna.create_study(direction='minimize',
                                 sampler=optuna.samplers.TPESampler(seed=seed))
    seeded.optimize(objective, n_trials=BUDGET)
    optuna_runs.append([trial.value for trial in seeded.trials])
    optuna_points.append(np.array([[t.params['alpha'], t.params['l1_ratio']]
                                   for t in seeded.trials]))

# TPESampler первые n_startup_trials испытаний сэмплирует случайно тем же генератором,
# что и наш random_search, поэтому при одинаковом зерне стартовые точки совпадают буквально
startup = optuna.samplers.TPESampler()._n_startup_trials
print(f'Стартовая случайная фаза TPESampler: {startup} испытаний')
print(f'Первые {startup} точек random_search и optuna совпадают:',
      all(np.allclose(a[:startup], b[:startup]) for a, b in zip(random_points, optuna_points)))

Стартовая случайная фаза TPESampler: 10 испытаний
Первые 10 точек random_search и optuna совпадают: True


In [410]:
grid_run = grid_history['score'].tolist()

# Эталон считаем НЕЗАВИСИМО от сравниваемых методов - плотной сеткой 15 x 9.
# Иначе порог задавал бы один из участников сравнения (и это был бы grid search)
dense_alphas = np.logspace(np.log10(ALPHA_LOW), np.log10(ALPHA_HIGH), 15)
dense_l1 = np.linspace(L1_LOW, L1_HIGH, 9)
started = time.time()
reference = min(valid_mae(alpha, l1_ratio) for alpha in dense_alphas for l1_ratio in dense_l1)
print(f'Эталон плотной сетки {len(dense_alphas)}x{len(dense_l1)} = '
      f'{len(dense_alphas) * len(dense_l1)} испытаний за {time.time() - started:.1f} с: '
      f'MAE = {reference:.4f}')


def trials_to_reach(values, level, budget=BUDGET):
    """Сколько испытаний нужно, чтобы впервые достичь уровня level.

    Запуск, который так и не достиг уровня, считаем цензурированным и штрафуем
    значением budget + 1: молча выбрасывать такие запуски нельзя - среднее по одним
    лишь удачным запускам делает метод тем лучше, чем чаще он проваливается
    """
    for i, value in enumerate(best_so_far(values), start=1):
        if value <= level:
            return i
    return budget + 1


rows = []
for name, runs in [('Grid search', [grid_run]),
                   ('Random search', random_runs),
                   ('Optuna (TPE)', optuna_runs)]:
    curves = np.array([best_so_far(run) for run in runs])
    reached = [trials_to_reach(run, reference * 1.001) for run in runs]
    rows.append({
        'метод': name,
        'лучший MAE': curves[:, -1].mean(),
        'MAE после 10 испытаний': curves[:, 9].mean(),
        'MAE после 20 испытаний': curves[:, 19].mean(),
        'запусков, достигших эталона': f'{sum(r <= BUDGET for r in reached)} из {len(runs)}',
        'испытаний до эталона +0.1%': float(np.mean(reached)),
        'доля испытаний в хорошей зоне': float(np.mean([np.mean(np.array(run) <= reference * 1.005)
                                                        for run in runs])),
    })
budget_table = pd.DataFrame(rows).set_index('метод')
budget_table.round(3)

Эталон плотной сетки 15x9 = 135 испытаний за 13.2 с: MAE = 692.1192


,лучший MAE,MAE после 10 испытаний,MAE после 20 испытаний,"запусков, достигших эталона",испытаний до эталона +0.1%,доля испытаний в хорошей зоне
метод,,,,,,
Grid search,692.119,692.655,692.573,1 из 1,2.0,0.250
Random search,692.967,695.311,694.876,2 из 5,28.8,0.120
Optuna (TPE),692.645,695.311,692.693,5 из 5,14.4,0.505


**Сравнение подходов.**

* **Grid search** гарантирует лучшую точку **внутри сетки** и здесь действительно находит
  глобальный оптимум - но лишь потому, что оптимальная точка случайно совпала с узлом сетки.
  Обратите внимание: по колонке «испытаний до эталона» сетку сравнивать с остальными нельзя -
  эталоном служит фактически её же собственный узел. Зато хорошо видно другое: сетка **вслепую
  тратит бюджет**, три четверти её испытаний приходится на заведомо плохую область больших
  `alpha`, а с ростом числа гиперпараметров число узлов растёт лавинообразно.
* **Random search** при равном бюджете исследует пространство равномерно, но не помнит
  предыдущих испытаний. Именно здесь видно преимущество байесовской оптимизации в чистом виде:
  случайный поиск дошёл до эталонного значения далеко не во всех запусках, тогда как `optuna` -
  во всех, и в среднем ей потребовалось на это заметно меньше испытаний. Запуск, так и не
  достигший цели, мы штрафуем значением «бюджет + 1»: выбрасывать провалы из среднего нельзя,
  иначе метод оказался бы тем «быстрее», чем чаще он проваливается.
* **Optuna (TPE)** использует историю: после стартовой случайной фазы она концентрирует испытания
  в перспективной области. В таблице это видно по колонке «доля испытаний в хорошей зоне» - она
  у `optuna` заметно выше, чем у случайного поиска и у сетки. Это и означает «приблизиться
  к оптимуму за меньшее число операций»: при дорогой целевой функции (а полная кросс-валидация
  дороже одного обучения в `k` раз) экономия испытаний прямо превращается в экономию времени.

Сравнение случайного поиска и `optuna` получилось особенно чистым: `TPESampler` первые
`n_startup_trials = 10` испытаний сэмплирует случайно, причём тем же генератором и теми же
преобразованиями, что и наша функция `random_search`. При одинаковом зерне **первые 10 точек
у обоих методов совпадают буквально** - отсюда и одинаковая колонка «MAE после 10 испытаний».
Значит, всё расхождение после десятого испытания - это чистый вклад модели TPE, а не удачный старт.

Честная оговорка: на нашей задаче поверхность качества почти плоская - все три метода приходят
к MAE около 692, и разница между ними исчисляется десятыми долями доллара. Преимущество
байесовской оптимизации проявляется тем сильнее, чем больше гиперпараметров и чем дороже одно
испытание.

### ⬇️ Задание 7.6. *Запустите optuna на одной из схем кросс-валидации.*

Оценка по одной валидационной выборке зависит от того, как именно прошло разбиение. Сделаем её
честнее: вместо одной валидации усредняем MAE по фолдам **Stratified K-Fold** - схемы, выбранной
в задании 5.4. Два принципиальных момента:

* `MinMaxScaler` обучаем **внутри каждого фолда**, иначе статистики тестового фолда протекут
  в обучение;
* фолды строим **только по train + valid**; тестовый период не участвует в подборе вообще,
  иначе подбор гиперпараметров подглядывал бы в тест.

In [411]:
cv_pool = data[data[DATE_FIELD] < TEST_DATE].reset_index(drop=True)
X_pool = cv_pool[MODEL_FEATURES]
y_pool = cv_pool[TARGET].to_numpy()
cv_folds = my_stratified_kfold(cv_pool, K, stratify_field='price_bin', shuffle=True)
print(f'Пул для кросс-валидации: {len(cv_pool)} строк (train + valid), '
      f'тест из {len(test_part)} строк не используется.')


def objective_cv(trial):
    """Целевая функция: средний MAE по фолдам кросс-валидации"""
    alpha = trial.suggest_float('alpha', ALPHA_LOW, ALPHA_HIGH, log=True)
    l1_ratio = trial.suggest_float('l1_ratio', L1_LOW, L1_HIGH)
    scores = []
    for train_idx, test_idx in cv_folds:
        fold_scaler = MinMaxScaler().fit(X_pool.iloc[train_idx])
        model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio,
                           random_state=RANDOM_STATE, max_iter=5000)
        model.fit(fold_scaler.transform(X_pool.iloc[train_idx]), y_pool[train_idx])
        scores.append(mean_absolute_error(
            y_pool[test_idx], model.predict(fold_scaler.transform(X_pool.iloc[test_idx]))))
    return float(np.mean(scores))


started = time.time()
study_cv = optuna.create_study(direction='minimize',
                               sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_cv.optimize(objective_cv, n_trials=20)
print(f'Optuna + Stratified 5-fold CV: 20 испытаний за {time.time() - started:.1f} с')
print(f'Лучший средний MAE по фолдам = {study_cv.best_value:.4f}, '
      f'параметры: {study_cv.best_params}')

Пул для кросс-валидации: 38703 строк (train + valid), тест из 9676 строк не используется.
Optuna + Stratified 5-fold CV: 20 испытаний за 17.3 с
Лучший средний MAE по фолдам = 688.3444, параметры: {'alpha': 0.00010323169081450829, 'l1_ratio': 0.8610498980078878}


In [412]:
def test_mae_for(alpha, l1_ratio):
    """Обучает модель на всём пуле train+valid и меряет MAE на нетронутом тесте"""
    pool_scaler = MinMaxScaler().fit(X_pool)
    model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio,
                       random_state=RANDOM_STATE, max_iter=5000)
    model.fit(pool_scaler.transform(X_pool), y_pool)
    return mean_absolute_error(test_part[TARGET],
                               model.predict(pool_scaler.transform(test_part[MODEL_FEATURES])))


# Сравниваем два СПОСОБА ПОДБОРА параметров на одной и той же честной метрике —
# MAE на отложенном тесте, которого не видел ни один из них
comparison_cv = pd.DataFrame([
    {'способ подбора': 'одна валидация (out-of-time)',
     'alpha': study.best_params['alpha'], 'l1_ratio': study.best_params['l1_ratio'],
     'внутренняя оценка MAE': study.best_value,
     'MAE на отложенном тесте': test_mae_for(study.best_params['alpha'],
                                             study.best_params['l1_ratio'])},
    {'способ подбора': 'Stratified 5-fold CV',
     'alpha': study_cv.best_params['alpha'], 'l1_ratio': study_cv.best_params['l1_ratio'],
     'внутренняя оценка MAE': study_cv.best_value,
     'MAE на отложенном тесте': test_mae_for(study_cv.best_params['alpha'],
                                             study_cv.best_params['l1_ratio'])},
]).set_index('способ подбора')
comparison_cv.round(4)

,alpha,l1_ratio,внутренняя оценка MAE,MAE на отложенном тесте
способ подбора,,,,
одна валидация (out-of-time),0.0001,0.8937,692.6537,690.9043
Stratified 5-fold CV,0.0001,0.8610,688.3444,690.9345


Два числа в колонке «внутренняя оценка MAE» между собой **не сравнимы**: первое получено на одной
out-of-time валидации, второе - это среднее по случайным стратифицированным фолдам пула
train + valid, то есть другая выборка. Сравнивать способы подбора нужно по общей честной метрике -
по колонке **«MAE на отложенном тесте»**, которого не видел ни один из них. Там разница
составляет доли доллара: на такой плоской поверхности качества оба способа приводят практически
к одной и той же модели.

Ценность кросс-валидации не в том, что она даёт лучшее число, а в том, что её оценка опирается на
все данные и почти не зависит от удачности одного разбиения. Платим за это ростом стоимости в `k`
раз. Это и есть тот честный режим, о котором говорит введение: подбор гиперпараметров и отбор
признаков стоит сочетать с кросс-валидацией, чтобы модель не переобучалась под конкретную
валидационную выборку.